# RideBase — 01 Veri Setini Tanıma

Bu notebook RideBase sentetik motosiklet bakım veri setinin dosya yapısını,
tablolarını, sütunlarını, veri tiplerini ve temel boyutlarını incelemek için
hazırlanmıştır.

Bu aşamada veri temizleme, EDA veya modelleme yapılmamaktadır.

## 1. Kütüphaneler ve Görüntüleme Ayarları

In [1]:
from pathlib import Path
import os
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 2. Path Tanımları

Notebook, çalıştırıldığı bilgisayardan bağımsız olsun diye hard-coded
absolute path kullanmıyor. Proje kökü, notebookun çalışma dizininden
(`notebooks/`) yukarı doğru proje işaretlerini arayarak bulunuyor. Analiz kaynağı sibling `ridebase_v1_2` release köküdür; `ridebase-ml/data` altındaki legacy kopya kullanılmaz.

In [2]:
def find_project_root(markers=("data", "src", "notebooks")) -> Path:
    """notebooks/ klasöründen yukarı çıkarak data/ ve src/ içeren proje kökünü bulur."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return current.parent  # notebooks/ içinden calistirildiginda güvenli fallback


PROJECT_ROOT = find_project_root()
DATASET_VERSION = "1.2.0"
DATASET_ROOT = PROJECT_ROOT.parent / "ridebase_v1_2"
RAW_DIR = DATASET_ROOT / "source_tables"
PROCESSED_DIR = DATASET_ROOT / "derived_outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"

if not RAW_DIR.exists() or not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"RideBase v1.2 bulunamadı: {DATASET_ROOT}")

print("Project Root:", PROJECT_ROOT)
print("Dataset Root (v1.2):", DATASET_ROOT)
print("Raw Data:", RAW_DIR)
print("Processed Data:", PROCESSED_DIR)

Project Root: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml
Dataset Root (v1.2): /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase_v1_2
Raw Data: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase_v1_2/source_tables
Processed Data: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase_v1_2/derived_outputs


## 3. Dosya Envanteri Doğrulama

Beklenen dosya listesi ile `ridebase_v1_2/source_tables/` ve
`ridebase_v1_2/derived_outputs/` klasörlerindeki gerçek dosyalar
karşılaştırılıyor. Eksik veya fazla dosya varsa burada raporlanır.

In [3]:
EXPECTED_SOURCE_FILES = [
    "workshops.csv",
    "customers.csv",
    "motorcycles.csv",
    "ridebase_motorcycle_models_v1.csv",
    "usage_profiles.csv",
    "mileage_timeline_monthly.csv",
    "maintenance_tasks.csv",
    "maintenance_policies.csv",
    "appointments.csv",
    "services.csv",
    "services_enriched.csv",
    "service_tasks.csv",
    "service_parts.csv",
]

EXPECTED_DERIVED_FILES = [
    "service_status_history.csv",
    "noise_audit.csv",
    "ml_maintenance_snapshots.parquet",
    "ml_next_service_targets.parquet",
    "ml_next_task_targets.parquet",
    "split_manifest.csv",
    "dataset_metadata.json",
    "quality_report.json",
]

actual_source_files = sorted(p.name for p in RAW_DIR.iterdir() if p.is_file())
actual_derived_files = sorted(p.name for p in PROCESSED_DIR.iterdir() if p.is_file())

missing_source = sorted(set(EXPECTED_SOURCE_FILES) - set(actual_source_files))
extra_source = sorted(set(actual_source_files) - set(EXPECTED_SOURCE_FILES))
missing_derived = sorted(set(EXPECTED_DERIVED_FILES) - set(actual_derived_files))
extra_derived = sorted(set(actual_derived_files) - set(EXPECTED_DERIVED_FILES))

print("=== source_tables ===")
print("Bulunan dosya sayısı:", len(actual_source_files))
print("Eksik dosyalar:", missing_source if missing_source else "YOK")
print("Fazla dosyalar:", extra_source if extra_source else "YOK")

print("\n=== derived_outputs ===")
print("Bulunan dosya sayısı:", len(actual_derived_files))
print("Eksik dosyalar:", missing_derived if missing_derived else "YOK")
print("Fazla dosyalar:", extra_derived if extra_derived else "YOK")

=== source_tables ===
Bulunan dosya sayısı: 13
Eksik dosyalar: YOK
Fazla dosyalar: YOK

=== derived_outputs ===
Bulunan dosya sayısı: 8
Eksik dosyalar: YOK
Fazla dosyalar: YOK


## 4. Tüm DataFrame'leri Yükleme

Her kaynak tablo ve türetilmiş (derived) çıktı, ayrı ve açıkça isimlendirilmiş
bir DataFrame değişkenine yükleniyor. CSV dosyaları `utf-8-sig` kodlamasıyla
okunuyor çünkü bazı kaynak tablolarda BOM (byte order mark) karakteri var.

### 4.1 Kaynak Tablolar (`data/raw/source_tables/`)

In [4]:
df_workshops = pd.read_csv(RAW_DIR / "workshops.csv", encoding="utf-8-sig")
df_customers = pd.read_csv(RAW_DIR / "customers.csv", encoding="utf-8-sig")
df_motorcycles = pd.read_csv(RAW_DIR / "motorcycles.csv", encoding="utf-8-sig")
df_motorcycle_models = pd.read_csv(RAW_DIR / "ridebase_motorcycle_models_v1.csv", encoding="utf-8-sig")
df_usage_profiles = pd.read_csv(RAW_DIR / "usage_profiles.csv", encoding="utf-8-sig")
df_mileage_monthly = pd.read_csv(RAW_DIR / "mileage_timeline_monthly.csv", encoding="utf-8-sig")
df_maintenance_tasks = pd.read_csv(RAW_DIR / "maintenance_tasks.csv", encoding="utf-8-sig")
df_maintenance_policies = pd.read_csv(RAW_DIR / "maintenance_policies.csv", encoding="utf-8-sig")
df_appointments = pd.read_csv(RAW_DIR / "appointments.csv", encoding="utf-8-sig")
df_services = pd.read_csv(RAW_DIR / "services.csv", encoding="utf-8-sig")
df_services_enriched = pd.read_csv(RAW_DIR / "services_enriched.csv", encoding="utf-8-sig")
df_service_tasks = pd.read_csv(RAW_DIR / "service_tasks.csv", encoding="utf-8-sig")
df_service_parts = pd.read_csv(RAW_DIR / "service_parts.csv", encoding="utf-8-sig")

print("Kaynak tablolar yüklendi.")

Kaynak tablolar yüklendi.


### 4.2 Derived / ML Çıktıları (`data/processed/derived_outputs/`)

In [5]:
df_service_status_history = pd.read_csv(PROCESSED_DIR / "service_status_history.csv", encoding="utf-8-sig")
df_noise_audit = pd.read_csv(PROCESSED_DIR / "noise_audit.csv", encoding="utf-8-sig")
df_split_manifest = pd.read_csv(PROCESSED_DIR / "split_manifest.csv", encoding="utf-8-sig")

df_ml_snapshots = pd.read_parquet(PROCESSED_DIR / "ml_maintenance_snapshots.parquet")
df_next_service_targets = pd.read_parquet(PROCESSED_DIR / "ml_next_service_targets.parquet")
df_next_task_targets = pd.read_parquet(PROCESSED_DIR / "ml_next_task_targets.parquet")

print("Derived / ML çıktıları yüklendi.")

Derived / ML çıktıları yüklendi.


### 4.3 JSON Metadata Dosyaları

In [6]:
with open(PROCESSED_DIR / "dataset_metadata.json", encoding="utf-8") as f:
    dataset_metadata = json.load(f)

with open(PROCESSED_DIR / "quality_report.json", encoding="utf-8") as f:
    quality_report = json.load(f)

loaded_dataset_version = dataset_metadata["dataset"]["dataset_version"]
if loaded_dataset_version != DATASET_VERSION:
    raise RuntimeError(
        f"Dataset version mismatch: beklenen={DATASET_VERSION}, bulunan={loaded_dataset_version}"
    )

print("Dataset version validation: PASS ->", loaded_dataset_version)
print("dataset_metadata.json anahtarları:")
print(list(dataset_metadata.keys()))
print("\nquality_report.json anahtarları:")
print(list(quality_report.keys()))

Dataset version validation: PASS -> 1.2.0
dataset_metadata.json anahtarları:
['dataset', 'design_principles', 'prediction_tasks', 'coverage', 'ml_snapshot_contract', 'split_contract', 'provenance', 'file_catalog', 'expected_final_support_files', 'known_v1_notes', 'changelog_summary', 'null_semantics', 'layer_definitions', 'target_semantics', 'noise_audit_semantics', 'appointment_lifecycle', 'workshop_variability_decision', 'failure_generation_model']

quality_report.json anahtarları:
['report', 'executive_summary', 'reliability_realism_checks', 'quality_checks']


## 5. DataFrame Registry

Tüm DataFrame'ler, ileride otomatik işlem yapabilmek için tek bir sözlükte
toplanıyor. Ayrıca her tablonun hangi dosyadan, hangi formatta ve hangi
klasörden geldiğini tutan bir meta veri listesi oluşturuluyor
(`dataset_inventory`, `column_dictionary` vb. tablolar bu listeden üretilecek).

In [7]:
dataframes = {
    "workshops": df_workshops,
    "customers": df_customers,
    "motorcycles": df_motorcycles,
    "motorcycle_models": df_motorcycle_models,
    "usage_profiles": df_usage_profiles,
    "mileage_monthly": df_mileage_monthly,
    "maintenance_tasks": df_maintenance_tasks,
    "maintenance_policies": df_maintenance_policies,
    "appointments": df_appointments,
    "services": df_services,
    "services_enriched": df_services_enriched,
    "service_tasks": df_service_tasks,
    "service_parts": df_service_parts,
    "service_status_history": df_service_status_history,
    "noise_audit": df_noise_audit,
    "ml_snapshots": df_ml_snapshots,
    "next_service_targets": df_next_service_targets,
    "next_task_targets": df_next_task_targets,
    "split_manifest": df_split_manifest,
}

# key: registry anahtari, var: degisken adi (string), file_name/file_type/source: kaynak bilgisi
table_meta = [
    {"key": "workshops", "var": "df_workshops", "file_name": "workshops.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "customers", "var": "df_customers", "file_name": "customers.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "motorcycles", "var": "df_motorcycles", "file_name": "motorcycles.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "motorcycle_models", "var": "df_motorcycle_models", "file_name": "ridebase_motorcycle_models_v1.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "usage_profiles", "var": "df_usage_profiles", "file_name": "usage_profiles.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "mileage_monthly", "var": "df_mileage_monthly", "file_name": "mileage_timeline_monthly.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "maintenance_tasks", "var": "df_maintenance_tasks", "file_name": "maintenance_tasks.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "maintenance_policies", "var": "df_maintenance_policies", "file_name": "maintenance_policies.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "appointments", "var": "df_appointments", "file_name": "appointments.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "services", "var": "df_services", "file_name": "services.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "services_enriched", "var": "df_services_enriched", "file_name": "services_enriched.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "service_tasks", "var": "df_service_tasks", "file_name": "service_tasks.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "service_parts", "var": "df_service_parts", "file_name": "service_parts.csv", "file_type": "csv", "source": "raw/source_tables"},
    {"key": "service_status_history", "var": "df_service_status_history", "file_name": "service_status_history.csv", "file_type": "csv", "source": "processed/derived_outputs"},
    {"key": "noise_audit", "var": "df_noise_audit", "file_name": "noise_audit.csv", "file_type": "csv", "source": "processed/derived_outputs"},
    {"key": "ml_snapshots", "var": "df_ml_snapshots", "file_name": "ml_maintenance_snapshots.parquet", "file_type": "parquet", "source": "processed/derived_outputs"},
    {"key": "next_service_targets", "var": "df_next_service_targets", "file_name": "ml_next_service_targets.parquet", "file_type": "parquet", "source": "processed/derived_outputs"},
    {"key": "next_task_targets", "var": "df_next_task_targets", "file_name": "ml_next_task_targets.parquet", "file_type": "parquet", "source": "processed/derived_outputs"},
    {"key": "split_manifest", "var": "df_split_manifest", "file_name": "split_manifest.csv", "file_type": "csv", "source": "processed/derived_outputs"},
]

print("Registry içindeki tablo sayısı:", len(dataframes))
print(list(dataframes.keys()))

Registry içindeki tablo sayısı: 19
['workshops', 'customers', 'motorcycles', 'motorcycle_models', 'usage_profiles', 'mileage_monthly', 'maintenance_tasks', 'maintenance_policies', 'appointments', 'services', 'services_enriched', 'service_tasks', 'service_parts', 'service_status_history', 'noise_audit', 'ml_snapshots', 'next_service_targets', 'next_task_targets', 'split_manifest']


## 6. Her Tablonun Sütunları, Head, Dtypes, Info ve Eksik/Duplicate Sayıları

Her tablo için: shape, **tüm** sütun adları, `head()`, `dtypes`, `info()`,
toplam eksik (missing) değer sayısı ve duplicate satır sayısı gösteriliyor.
Tekrarı azaltmak için tek bir `profile_dataframe` yardımcı fonksiyonu
tanımlanıyor, ancak her tablo kendi başlığı altında **açıkça isimlendirilmiş
DataFrame değişkeni** ile ayrı ayrı çağrılıyor.

In [8]:
def profile_dataframe(df: pd.DataFrame, name: str) -> None:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("\nSütunlar (tümü, {} adet):".format(df.shape[1]))
    print(df.columns.tolist())
    print("\ndtypes:")
    print(df.dtypes)
    print("\ninfo():")
    df.info()
    print("\nToplam missing value:", int(df.isna().sum().sum()))
    print("Duplicate row sayısı:", int(df.duplicated().sum()))
    print("\nhead():")
    display(df.head())

### workshops.csv

In [9]:
profile_dataframe(df_workshops, "workshops.csv")

=== workshops.csv ===
Shape: (10, 35)

Sütunlar (tümü, 35 adet):
['workshop_id', 'workshop_name', 'city', 'district_profile', 'region', 'climate_zone', 'price_level', 'opening_time', 'closing_time', 'working_days', 'sunday_open', 'service_bay_count', 'technician_count', 'senior_technician_count', 'daily_service_capacity', 'appointment_rate', 'walk_in_rate', 'avg_parts_lead_days', 'same_day_parts_rate', 'multibrand', 'specialization', 'supported_powertrains', 'supported_final_drives', 'supports_ev', 'supports_abs_diagnostics', 'supports_advanced_diagnostics', 'labor_rate_index', 'parts_sale_multiplier', 'customer_volume_index', 'courier_customer_share', 'seasonality_strength', 'data_origin', 'generator_version', 'scenario_id', 'notes']

dtypes:
workshop_id                       object
workshop_name                     object
city                              object
district_profile                  object
region                            object
climate_zone                      object


,workshop_id,workshop_name,city,district_profile,region,climate_zone,price_level,opening_time,closing_time,working_days,sunday_open,service_bay_count,technician_count,senior_technician_count,daily_service_capacity,appointment_rate,walk_in_rate,avg_parts_lead_days,same_day_parts_rate,multibrand,specialization,supported_powertrains,supported_final_drives,supports_ev,supports_abs_diagnostics,supports_advanced_diagnostics,labor_rate_index,parts_sale_multiplier,customer_volume_index,courier_customer_share,seasonality_strength,data_origin,generator_version,scenario_id,notes
0,WS0001,RideBase Sentetik Servis İstanbul Anadolu,İstanbul,METROPOLITAN_DENSE,Marmara,TEMPERATE_HUMID,LOW,08:30,18:30,"MON,TUE,WED,THU,FRI,SAT",1,4,5,1,11,0.45,0.55,2.0,0.72,1,SCOOTER;CUB;COMMUTER,ICE;EV,CHAIN;V_BELT;DIRECT,1,1,0,0.90,0.92,1.25,0.28,0.12,SYNTHETIC,1.2.0,URBAN_VALUE_HIGH_VOLUME,"Yüksek hacimli, fiyat hassasiyeti yüksek şehir içi servis profili."
1,WS0002,RideBase Sentetik Servis Ankara,Ankara,METROPOLITAN,İç Anadolu,CONTINENTAL_DRY,MEDIUM,08:30,18:00,"MON,TUE,WED,THU,FRI,SAT",0,6,7,2,15,0.58,0.42,1.8,0.78,1,NAKED;SCOOTER;TOURING,ICE;EV,CHAIN;V_BELT;BELT;DIRECT,1,1,1,1.00,1.00,1.05,0.18,0.25,SYNTHETIC,1.2.0,BALANCED_MULTIBRAND,Dengeli randevu/yaya geliş oranı olan çok markalı merkez servis.
2,WS0003,RideBase Sentetik Servis İzmir,İzmir,COASTAL_METROPOLITAN,Ege,MEDITERRANEAN_COASTAL,MEDIUM,09:00,19:00,"MON,TUE,WED,THU,FRI,SAT",0,7,8,2,18,0.62,0.38,1.5,0.82,1,SCOOTER;BIG_SCOOTER;NAKED,ICE;EV,CHAIN;V_BELT;BELT;DIRECT,1,1,1,1.02,1.03,1.18,0.22,0.10,SYNTHETIC,1.2.0,COASTAL_SCOOTER_HEAVY,Ilıman iklim nedeniyle yıl boyu daha yüksek scooter kullanımına göre tasarlanmıştır.
3,WS0004,RideBase Sentetik Servis Bursa,Bursa,INDUSTRIAL_METROPOLITAN,Marmara,TEMPERATE_HUMID,HIGH,08:00,18:30,"MON,TUE,WED,THU,FRI,SAT",0,8,10,3,21,0.70,0.30,1.3,0.86,1,NAKED;SPORT;TOURING;BIG_SCOOTER,ICE;EV,CHAIN;V_BELT;BELT;SHAFT;DIRECT,1,1,1,1.10,1.10,1.12,0.12,0.16,SYNTHETIC,1.2.0,TECHNICAL_HIGH_CAPACITY,Daha yüksek teknik kapasite ve ileri teşhis imkânına sahip servis profili.
4,WS0005,RideBase Sentetik Premium Servis Antalya,Antalya,COASTAL_TOURISM,Akdeniz,MEDITERRANEAN_HOT,PREMIUM,09:00,19:00,"MON,TUE,WED,THU,FRI,SAT",0,9,11,4,23,0.78,0.22,1.2,0.90,1,BIG_SCOOTER;TOURING;ADVENTURE;PREMIUM,ICE;EV,CHAIN;V_BELT;BELT;SHAFT;DIRECT,1,1,1,1.22,1.18,1.08,0.08,0.08,SYNTHETIC,1.2.0,PREMIUM_TOURING_COASTAL,"Premium, tur ve büyük scooter ağırlıklı kıyı servisi."


### customers.csv

In [10]:
profile_dataframe(df_customers, "customers.csv")

=== customers.csv ===
Shape: (7000, 16)

Sütunlar (tümü, 16 adet):
['customer_id', 'workshop_id', 'customer_type', 'first_seen_date', 'city', 'region', 'is_fleet_customer', 'preferred_contact', 'acquisition_channel', 'synthetic_alias', 'is_active', 'churn_date', 'data_origin', 'generator_version', 'random_seed', 'notes']

dtypes:
customer_id             object
workshop_id             object
customer_type           object
first_seen_date         object
city                    object
region                  object
is_fleet_customer        int64
preferred_contact       object
acquisition_channel     object
synthetic_alias         object
is_active                int64
churn_date              object
data_origin             object
generator_version       object
random_seed              int64
notes                  float64
dtype: object

info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 16 columns):
 #   Column               Non-Null Count  

,customer_id,workshop_id,customer_type,first_seen_date,city,region,is_fleet_customer,preferred_contact,acquisition_channel,synthetic_alias,is_active,churn_date,data_origin,generator_version,random_seed,notes
0,C000001,WS0003,INDIVIDUAL,2024-06-08,İzmir,Ege,0,WHATSAPP,GOOGLE_MAPS,Customer_000001,1,NaN,SYNTHETIC,1.2.0,42,NaN
1,C000002,WS0008,INDIVIDUAL,2023-07-10,Gaziantep,Güneydoğu Anadolu,0,WHATSAPP,WALK_IN,Customer_000002,1,NaN,SYNTHETIC,1.2.0,42,NaN
2,C000003,WS0008,INDIVIDUAL,2023-06-22,Gaziantep,Güneydoğu Anadolu,0,WHATSAPP,REFERRAL,Customer_000003,1,NaN,SYNTHETIC,1.2.0,42,NaN
3,C000004,WS0009,INDIVIDUAL,2025-09-19,Samsun,Karadeniz,0,WHATSAPP,REFERRAL,Customer_000004,1,NaN,SYNTHETIC,1.2.0,42,NaN
4,C000005,WS0007,FLEET,2022-01-22,Adana,Akdeniz,1,WHATSAPP,REFERRAL,Customer_000005,1,NaN,SYNTHETIC,1.2.0,42,NaN


### motorcycles.csv

In [11]:
profile_dataframe(df_motorcycles, "motorcycles.csv")

=== motorcycles.csv ===
Shape: (10000, 29)

Sütunlar (tümü, 29 adet):
['motorcycle_id', 'customer_id', 'workshop_id', 'model_id', 'brand', 'model_name', 'category', 'powertrain_type', 'production_year', 'production_year_basis', 'first_registration_date', 'ownership_start_date', 'initial_mileage_km', 'initial_mileage_basis', 'observation_start_date', 'observation_end_date', 'vin_synthetic', 'plate_synthetic', 'is_active', 'data_origin', 'generator_version', 'random_seed', 'model_assignment_basis', 'notes', 'brand_risk_residual', 'model_risk_residual', 'chronic_risk_tier', 'warranty_months', 'warranty_km_limit']

dtypes:
motorcycle_id               object
customer_id                 object
workshop_id                 object
model_id                    object
brand                       object
model_name                  object
category                    object
powertrain_type             object
production_year              int64
production_year_basis       object
first_registration_date

,motorcycle_id,customer_id,workshop_id,model_id,brand,model_name,category,powertrain_type,production_year,production_year_basis,first_registration_date,ownership_start_date,initial_mileage_km,initial_mileage_basis,observation_start_date,observation_end_date,vin_synthetic,plate_synthetic,is_active,data_origin,generator_version,random_seed,model_assignment_basis,notes,brand_risk_residual,model_risk_residual,chronic_risk_tier,warranty_months,warranty_km_limit
0,MC000001,C000001,WS0003,YAMAHA_NMAX125,Yamaha,NMAX 125,SCOOTER,ICE,2017,V1_1_SIMULATION_MODEL_RANGE_VALIDATED,2017-02-16,2019-01-05,83780,AGE_CUSTOMER_TYPE_CATEGORY_PRIOR,2024-06-21,NaN,RBSYNE3EFA77D23F0,SYN-35-000001,1,SYNTHETIC,1.2.0,42,MARKET_WEIGHT_X_CUSTOMER_TYPE_X_WORKSHOP_SPECIALIZATION,NaN,1.00172,0.98342,NORMAL,30,30000
1,MC000002,C000002,WS0008,HONDA_PCX125,Honda,PCX125,SCOOTER,ICE,2021,V1_1_SIMULATION_MODEL_RANGE_VALIDATED,2021-04-10,2023-03-19,27440,AGE_CUSTOMER_TYPE_CATEGORY_PRIOR,2023-07-12,NaN,RBSYN022AACC9F763,SYN-27-000002,1,SYNTHETIC,1.2.0,42,MARKET_WEIGHT_X_CUSTOMER_TYPE_X_WORKSHOP_SPECIALIZATION,NaN,1.00217,1.02866,NORMAL,24,30000
2,MC000003,C000003,WS0008,MONDIAL_50_UAG,Mondial,50 UAG,CUB,ICE,2020,V1_1_SIMULATION_MODEL_RANGE_VALIDATED,2020-04-22,2022-08-26,32650,AGE_CUSTOMER_TYPE_CATEGORY_PRIOR,2023-06-30,NaN,RBSYN28086615C98F,SYN-27-000003,1,SYNTHETIC,1.2.0,42,MARKET_WEIGHT_X_CUSTOMER_TYPE_X_WORKSHOP_SPECIALIZATION,NaN,1.00797,1.05773,NORMAL,24,40000
3,MC000004,C000004,WS0009,YAMAHA_XMAX250,Yamaha,XMAX 250,BIG_SCOOTER,ICE,2023,V1_1_SIMULATION_MODEL_RANGE_VALIDATED,2023-11-09,2024-02-12,11680,AGE_CUSTOMER_TYPE_CATEGORY_PRIOR,2025-09-25,NaN,RBSYN0E04DA1741F8,SYN-55-000004,1,SYNTHETIC,1.2.0,42,MARKET_WEIGHT_X_CUSTOMER_TYPE_X_WORKSHOP_SPECIALIZATION,NaN,1.00172,0.97849,NORMAL,36,40000
4,MC000005,C000005,WS0007,MONDIAL_TURISMO50I,Mondial,Turismo 50i,SCOOTER,ICE,2021,V1_1_SIMULATION_MODEL_RANGE_VALIDATED,2021-08-04,2021-09-16,13650,AGE_CUSTOMER_TYPE_CATEGORY_PRIOR,2022-01-30,NaN,RBSYN78C063F409D9,SYN-01-000005,1,SYNTHETIC,1.2.0,42,MARKET_WEIGHT_X_CUSTOMER_TYPE_X_WORKSHOP_SPECIALIZATION,NaN,1.00797,1.03336,NORMAL,30,30000


### ridebase_motorcycle_models_v1.csv

In [12]:
profile_dataframe(df_motorcycle_models, "ridebase_motorcycle_models_v1.csv")

=== ridebase_motorcycle_models_v1.csv ===
Shape: (39, 38)

Sütunlar (tümü, 38 adet):
['model_id', 'brand', 'model_name', 'variant_or_generation', 'category', 'market_priority', 'generator_weight_raw', 'generator_weight_seed', 'weight_basis', 'powertrain_type', 'engine_displacement_cc', 'cylinder_count', 'cooling_type', 'cooling_detail', 'final_drive_type', 'transmission_type', 'engine_oil_service_qty_l', 'oil_quantity_basis', 'spark_plug_count', 'fuel_type', 'production_start_year', 'production_end_year', 'spec_reference_year', 'policy_group', 'engine_oil_applicable', 'spark_plug_applicable', 'coolant_applicable', 'chain_maintenance_applicable', 'belt_maintenance_applicable', 'policy_ready', 'spec_confidence', 'source_type', 'primary_source_url', 'secondary_source_url', 'verification_notes', 'source_checked_at', 'production_year_range_basis', 'production_year_range_confidence']

dtypes:
model_id                             object
brand                                object
model_name  

,model_id,brand,model_name,variant_or_generation,category,market_priority,generator_weight_raw,generator_weight_seed,weight_basis,powertrain_type,engine_displacement_cc,cylinder_count,cooling_type,cooling_detail,final_drive_type,transmission_type,engine_oil_service_qty_l,oil_quantity_basis,spark_plug_count,fuel_type,production_start_year,production_end_year,spec_reference_year,policy_group,engine_oil_applicable,spark_plug_applicable,coolant_applicable,chain_maintenance_applicable,belt_maintenance_applicable,policy_ready,spec_confidence,source_type,primary_source_url,secondary_source_url,verification_notes,source_checked_at,production_year_range_basis,production_year_range_confidence
0,HONDA_PCX125,Honda,PCX125,current TR listing,SCOOTER,CORE,10.0,0.106270,ENGINEERING_PRIOR_NOT_MARKET_SHARE,ICE,125.0,1,LIQUID,NaN,V_BELT,CVT,0.8,2021 owner manual: after draining,1.0,GASOLINE,2015.0,2026.0,NaN,ICE_125_LIQUID_BELT,True,True,True,False,True,True,HIGH,OFFICIAL,https://www.honda.com.tr/motosiklet/modeller/scooter/honda-pcx125/ozellikler,https://www.motorhaber.com.tr/2025-yilinda-turkiye-de-en-cok-satan-motosikletler-190521,Core market row. 0.8 L oil figure is generation-specific; re-check before exact service-part quantity.,2026-08-25,SIMULATION_PRIOR_MODEL_RANGE,LOW
1,HONDA_ACTIVA125,Honda,Activa 125,TR listing,SCOOTER,CORE,8.0,0.085016,ENGINEERING_PRIOR_NOT_MARKET_SHARE,ICE,124.0,1,AIR,NaN,V_BELT,CVT,NaN,NaN,1.0,GASOLINE,2015.0,2026.0,NaN,ICE_125_SCOOTER_BELT,True,True,False,False,True,False,MEDIUM,OFFICIAL,https://www.honda.com.tr/motosiklet/modeller/scooter/lp/honda-activa-125-one-cikan-ozellikler,https://www.motorhaber.com.tr/2025-yilinda-turkiye-de-en-cok-satan-motosikletler-190521,Cooling and exact oil service quantity intentionally left unverified. v1.1 cooling type normalized to AIR from the model technical listing; retained as synthetic technical master data.,2026-08-25,SIMULATION_PRIOR_MODEL_RANGE,LOW
2,TVS_JUPITER125,TVS,Jupiter 125,current,SCOOTER,CORE,7.0,0.074389,ENGINEERING_PRIOR_NOT_MARKET_SHARE,ICE,124.8,1,AIR,NaN,V_BELT,CVT,NaN,NaN,1.0,GASOLINE,2015.0,2026.0,NaN,ICE_125_AIR_BELT,True,True,False,False,True,True,HIGH,OFFICIAL,https://www.tvsmotor.com/tvs-jupiter-125/disc,https://www.motorhaber.com.tr/2025-yilinda-turkiye-de-en-cok-satan-motosikletler-190521,"Official TVS spec: single-cylinder, air-cooled, 124.8 cc, CVT automatic.",2026-08-25,SIMULATION_PRIOR_MODEL_RANGE,LOW
3,KUBA_CG50_PRO,Kuba,CG50 Pro,current,CUB,CORE,5.0,0.053135,ENGINEERING_PRIOR_NOT_MARKET_SHARE,ICE,50.0,1,AIR,NaN,CHAIN,MANUAL_5,NaN,NaN,1.0,GASOLINE,2015.0,2026.0,NaN,ICE_50_AIR_CHAIN,True,True,False,True,False,True,MEDIUM,SECONDARY,https://www.epey.com/motosiklet/kuba-cg50-pro.html,https://www.motorhaber.com.tr/2025-yilinda-turkiye-de-en-cok-satan-motosikletler-190521,Secondary technical source; verify against manufacturer/manual before model-specific maintenance policy.,2026-08-25,SIMULATION_PRIOR_MODEL_RANGE,LOW
4,MONDIAL_WING50I,Mondial,Wing 50i,current,SCOOTER,CORE,5.0,0.053135,ENGINEERING_PRIOR_NOT_MARKET_SHARE,ICE,49.6,1,AIR,FAN_FORCED_AIR,V_BELT,AUTOMATIC,NaN,NaN,1.0,GASOLINE,2015.0,2026.0,NaN,ICE_50_AIR_BELT,True,True,False,False,True,True,HIGH,OFFICIAL,https://www.mondialmotor.com.tr/model/wing-50-i,https://www.motorhaber.com.tr/2025-yilinda-turkiye-de-en-cok-satan-motosikletler-190521,Official Mondial model page.,2026-08-25,SIMULATION_PRIOR_MODEL_RANGE,LOW


### usage_profiles.csv

In [13]:
profile_dataframe(df_usage_profiles, "usage_profiles.csv")

=== usage_profiles.csv ===
Shape: (10000, 30)

Sütunlar (tümü, 30 adet):
['usage_profile_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'usage_type', 'annual_km_baseline', 'annual_km_basis', 'city_ratio', 'highway_ratio', 'offroad_ratio', 'track_ratio', 'avg_ride_days_per_week', 'daily_active_probability', 'avg_km_per_active_day', 'weekday_multiplier', 'weekend_multiplier', 'riding_intensity', 'seasonality_strength', 'winter_usage_factor', 'weather_sensitivity', 'storage_condition', 'load_severity_factor', 'climate_zone', 'profile_start_date', 'profile_end_date', 'data_origin', 'generator_version', 'random_seed', 'profile_rule_version', 'notes']

dtypes:
usage_profile_id             object
motorcycle_id                object
customer_id                  object
workshop_id                  object
usage_type                   object
annual_km_baseline            int64
annual_km_basis              object
city_ratio                  float64
highway_ratio               float64
offroad_

,usage_profile_id,motorcycle_id,customer_id,workshop_id,usage_type,annual_km_baseline,annual_km_basis,city_ratio,highway_ratio,offroad_ratio,track_ratio,avg_ride_days_per_week,daily_active_probability,avg_km_per_active_day,weekday_multiplier,weekend_multiplier,riding_intensity,seasonality_strength,winter_usage_factor,weather_sensitivity,storage_condition,load_severity_factor,climate_zone,profile_start_date,profile_end_date,data_origin,generator_version,random_seed,profile_rule_version,notes
0,UP000001,MC000001,C000001,WS0003,COMMUTER,18507,OBSERVED_MONTHLY_EXPOSURE_ANNUALIZED_V1_2,0.9490,0.0462,0.0048,0.0000,5.70,0.770,71.4,1.151,0.851,HIGH,0.145,0.811,0.264,COVERED_PARKING,1.032,MEDITERRANEAN_COASTAL,2024-06-21,NaN,SYNTHETIC,1.2.0,42,1.2.0,NaN
1,UP000002,MC000002,C000002,WS0008,TOURING,13014,OBSERVED_MONTHLY_EXPOSURE_ANNUALIZED_V1_2,0.7390,0.2507,0.0103,0.0000,5.47,0.740,50.0,1.342,0.592,MEDIUM,0.144,0.784,0.412,COVERED_PARKING,1.014,SEMI_ARID_HOT,2023-07-12,NaN,SYNTHETIC,1.2.0,42,1.2.0,NaN
2,UP000003,MC000003,C000003,WS0008,WEEKEND,9259,OBSERVED_MONTHLY_EXPOSURE_ANNUALIZED_V1_2,0.7732,0.2167,0.0101,0.0000,3.96,0.536,47.5,1.197,0.591,MEDIUM,0.139,0.756,0.327,STREET,0.929,SEMI_ARID_HOT,2023-06-30,NaN,SYNTHETIC,1.2.0,42,1.2.0,NaN
3,UP000004,MC000004,C000004,WS0009,COMMUTER,4010,OBSERVED_MONTHLY_EXPOSURE_ANNUALIZED_V1_2,0.4634,0.4948,0.0202,0.0216,1.94,0.238,51.8,0.276,2.139,LOW,0.399,0.494,0.601,STREET,0.934,HUMID_RAINY,2025-09-25,NaN,SYNTHETIC,1.2.0,42,1.2.0,NaN
4,UP000005,MC000005,C000005,WS0007,COMMUTER,7340,OBSERVED_MONTHLY_EXPOSURE_ANNUALIZED_V1_2,0.5275,0.4420,0.0209,0.0096,2.77,0.323,62.7,0.651,1.441,LOW,0.515,0.486,0.780,GARAGE,0.924,MEDITERRANEAN_HOT,2022-01-30,NaN,SYNTHETIC,1.2.0,42,1.2.0,NaN


### mileage_timeline_monthly.csv

In [14]:
profile_dataframe(df_mileage_monthly, "mileage_timeline_monthly.csv")

=== mileage_timeline_monthly.csv ===
Shape: (344714, 15)

Sütunlar (tümü, 15 adet):
['timeline_id', 'motorcycle_id', 'workshop_id', 'year_month', 'period_start_date', 'period_end_date', 'opening_odometer_km', 'closing_odometer_km', 'km_added', 'active_days', 'avg_km_per_active_day_observed', 'usage_type', 'annual_km_baseline', 'data_origin', 'generator_version']

dtypes:
timeline_id                        object
motorcycle_id                      object
workshop_id                        object
year_month                         object
period_start_date                  object
period_end_date                    object
opening_odometer_km                 int64
closing_odometer_km                 int64
km_added                            int64
active_days                         int64
avg_km_per_active_day_observed    float64
usage_type                         object
annual_km_baseline                  int64
data_origin                        object
generator_version                  obj


Toplam missing value: 0


Duplicate row sayısı: 0

head():


,timeline_id,motorcycle_id,workshop_id,year_month,period_start_date,period_end_date,opening_odometer_km,closing_odometer_km,km_added,active_days,avg_km_per_active_day_observed,usage_type,annual_km_baseline,data_origin,generator_version
0,MTL0000001,MC000001,WS0003,2024-06,2024-06-21,2024-06-30,83780,84324,544,10,54.4,COMMUTER,18507,SYNTHETIC,1.2.0
1,MTL0000002,MC000001,WS0003,2024-07,2024-07-01,2024-07-31,84324,86225,1901,27,70.4,COMMUTER,18507,SYNTHETIC,1.2.0
2,MTL0000003,MC000001,WS0003,2024-08,2024-08-01,2024-08-31,86225,87883,1658,25,66.3,COMMUTER,18507,SYNTHETIC,1.2.0
3,MTL0000004,MC000001,WS0003,2024-09,2024-09-01,2024-09-30,87883,89357,1474,21,70.2,COMMUTER,18507,SYNTHETIC,1.2.0
4,MTL0000005,MC000001,WS0003,2024-10,2024-10-01,2024-10-31,89357,90793,1436,25,57.4,COMMUTER,18507,SYNTHETIC,1.2.0


### maintenance_tasks.csv

In [15]:
profile_dataframe(df_maintenance_tasks, "maintenance_tasks.csv")

=== maintenance_tasks.csv ===
Shape: (98, 17)

Sütunlar (tümü, 17 adet):
['task_code', 'canonical_name_tr', 'canonical_name_en', 'component_group', 'action_type', 'is_periodic', 'is_wear_based', 'is_fault_based', 'requires_part', 'applicable_powertrain', 'required_final_drive', 'required_cooling_type', 'required_transmission_type', 'part_category_hint', 'can_be_next_service_target', 'notes', 'taxonomy_version']

dtypes:
task_code                     object
canonical_name_tr             object
canonical_name_en             object
component_group               object
action_type                   object
is_periodic                    int64
is_wear_based                  int64
is_fault_based                 int64
requires_part                  int64
applicable_powertrain         object
required_final_drive          object
required_cooling_type         object
required_transmission_type    object
part_category_hint            object
can_be_next_service_target     int64
notes                

,task_code,canonical_name_tr,canonical_name_en,component_group,action_type,is_periodic,is_wear_based,is_fault_based,requires_part,applicable_powertrain,required_final_drive,required_cooling_type,required_transmission_type,part_category_hint,can_be_next_service_target,notes,taxonomy_version
0,ENGINE_OIL_CHECK,Motor Yağı Seviye Kontrolü,Engine Oil Level Check,ENGINE,INSPECT,1,0,0,0,ICE,NaN,NaN,NaN,ENGINE_OIL,1,Yağ seviyesi ve genel durumu kontrol edilir.,1.2.0
1,ENGINE_OIL_CHANGE,Motor Yağı Değişimi,Engine Oil Change,ENGINE,REPLACE,1,0,0,1,ICE,NaN,NaN,NaN,ENGINE_OIL,1,Yağ miktarı model teknik özelliğinden/politikadan türetilmelidir.,1.2.0
2,OIL_FILTER_CHANGE,Yağ Filtresi Değişimi,Oil Filter Replacement,ENGINE,REPLACE,1,0,0,1,ICE,NaN,NaN,NaN,OIL_FILTER,1,Yalnızca yağ filtresi bulunan uyumlu modellerde.,1.2.0
3,OIL_LEAK_INSPECTION,Motor Yağ Kaçağı Kontrolü,Engine Oil Leak Inspection,ENGINE,INSPECT,0,0,1,0,ICE,NaN,NaN,NaN,NaN,1,Arıza/şikâyet veya genel kontrol sonucu tetiklenebilir.,1.2.0
4,ENGINE_COMPRESSION_TEST,Motor Kompresyon Testi,Engine Compression Test,ENGINE,DIAGNOSE,0,0,1,0,ICE,NaN,NaN,NaN,NaN,1,"Performans düşüşü, zor çalışma veya motor arızası şüphesinde.",1.2.0


### maintenance_policies.csv

In [16]:
profile_dataframe(df_maintenance_policies, "maintenance_policies.csv")

=== maintenance_policies.csv ===
Shape: (621, 30)

Sütunlar (tümü, 30 adet):
['policy_id', 'scope_type', 'model_id', 'policy_group', 'task_code', 'policy_kind', 'initial_trigger_km', 'recurring_km', 'initial_trigger_months', 'recurring_months', 'trigger_mode', 'severe_use_multiplier', 'courier_multiplier', 'offroad_multiplier', 'dusty_environment_multiplier', 'wear_mean_km', 'wear_sd_km', 'wear_min_km', 'wear_max_km', 'precedence', 'evidence_level', 'source_authority', 'source_model_or_family', 'source_url', 'source_page_or_note', 'confidence', 'is_manufacturer_exact_interval', 'is_generator_active', 'policy_version', 'notes']

dtypes:
policy_id                          object
scope_type                         object
model_id                           object
policy_group                       object
task_code                          object
policy_kind                        object
initial_trigger_km                float64
recurring_km                      float64
initial_trigger_mont

,policy_id,scope_type,model_id,policy_group,task_code,policy_kind,initial_trigger_km,recurring_km,initial_trigger_months,recurring_months,trigger_mode,severe_use_multiplier,courier_multiplier,offroad_multiplier,dusty_environment_multiplier,wear_mean_km,wear_sd_km,wear_min_km,wear_max_km,precedence,evidence_level,source_authority,source_model_or_family,source_url,source_page_or_note,confidence,is_manufacturer_exact_interval,is_generator_active,policy_version,notes
0,POL0001,GROUP,NaN,EV_HUB_MOTOR,EV_TRACTION_BATTERY_HEALTH_CHECK,SCHEDULED,NaN,12000.0,NaN,12.0,WHICHEVER_FIRST,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,10,SIMULATION_PRIOR,RideBase simulation prior,NaN,NaN,NaN,LOW,0,1,1.2.0,Simulation cadence only; not claimed as manufacturer exact.
1,POL0002,GROUP,NaN,EV_HUB_MOTOR,EV_CHARGING_PORT_INSPECTION,SCHEDULED,NaN,6000.0,NaN,12.0,WHICHEVER_FIRST,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,10,SIMULATION_PRIOR,RideBase simulation prior,NaN,NaN,NaN,LOW,0,1,1.2.0,NaN
2,POL0003,GROUP,NaN,EV_HUB_MOTOR,EV_HIGH_VOLTAGE_SYSTEM_INSPECTION,SCHEDULED,NaN,12000.0,NaN,12.0,WHICHEVER_FIRST,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,10,SIMULATION_PRIOR,RideBase simulation prior,NaN,NaN,NaN,LOW,0,1,1.2.0,NaN
3,POL0004,GROUP,NaN,EV_HUB_MOTOR,EV_DRIVE_MOTOR_DIAGNOSTIC,CONDITION_BASED,NaN,NaN,NaN,NaN,CONDITION_ONLY,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,10,SIMULATION_PRIOR,RideBase simulation prior,NaN,NaN,NaN,LOW,0,1,1.2.0,NaN
4,POL0005,GROUP,NaN,EV_HUB_MOTOR,GENERAL_SAFETY_INSPECTION,SCHEDULED,NaN,6000.0,NaN,12.0,WHICHEVER_FIRST,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,10,SIMULATION_BACKOFF,RideBase engineering backoff,NaN,NaN,NaN,MEDIUM,0,1,1.2.0,NaN


### appointments.csv

In [17]:
profile_dataframe(df_appointments, "appointments.csv")

=== appointments.csv ===
Shape: (28150, 13)

Sütunlar (tümü, 13 adet):
['appointment_id', 'workshop_id', 'customer_id', 'motorcycle_id', 'created_at', 'scheduled_at', 'appointment_type', 'source', 'status', 'service_id', 'customer_request', 'data_origin', 'generator_version']

dtypes:
appointment_id       object
workshop_id          object
customer_id          object
motorcycle_id        object
created_at           object
scheduled_at         object
appointment_type     object
source               object
status               object
service_id           object
customer_request     object
data_origin          object
generator_version    object
dtype: object

info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28150 entries, 0 to 28149
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   appointment_id     28150 non-null  object
 1   workshop_id        28150 non-null  object
 2   customer_id        28150 

,appointment_id,workshop_id,customer_id,motorcycle_id,created_at,scheduled_at,appointment_type,source,status,service_id,customer_request,data_origin,generator_version
0,APT000001,WS0003,C000001,MC000001,2026-03-21 14:00:00,2026-03-25 10:43:00,PERIODIC,MOBILE_APP,CONVERTED_TO_SERVICE,SVC000003,PERIODIC,SYNTHETIC,1.2.0
1,APT000002,WS0003,C000001,MC000001,2026-06-12 15:45:00,2026-06-22 15:04:00,PERIODIC,PHONE,CONVERTED_TO_SERVICE,SVC000004,PERIODIC,SYNTHETIC,1.2.0
2,APT000003,WS0008,C000002,MC000002,2023-09-18 11:15:00,2023-09-19 12:20:00,REPAIR,WHATSAPP,CONVERTED_TO_SERVICE,SVC000005,REPAIR,SYNTHETIC,1.2.0
3,APT000004,WS0008,C000002,MC000002,2025-10-14 16:00:00,2025-10-17 15:18:00,PERIODIC,WHATSAPP,CONVERTED_TO_SERVICE,SVC000008,PERIODIC,SYNTHETIC,1.2.0
4,APT000005,WS0008,C000003,MC000003,2023-12-17 12:15:00,2023-12-20 16:44:00,PERIODIC,PHONE,CONVERTED_TO_SERVICE,SVC000009,PERIODIC,SYNTHETIC,1.2.0


### services.csv

In [18]:
profile_dataframe(df_services, "services.csv")

=== services.csv ===
Shape: (41518, 43)

Sütunlar (tümü, 43 adet):
['service_id', 'workshop_id', 'customer_id', 'motorcycle_id', 'appointment_id', 'service_type_code', 'primary_trigger_task', 'trigger_due_km', 'trigger_due_date', 'service_delay_days', 'arrival_mode', 'received_at', 'odometer_km', 'mileage_source', 'mileage_quality_flag', 'is_mileage_estimated', 'status', 'estimated_delivery_at', 'completed_at', 'delivered_at', 'customer_complaint', 'technician_notes', 'labor_minutes_pre_task', 'labor_total', 'parts_total', 'discount_total', 'tax_total', 'grand_total', 'is_breakdown', 'is_warranty', 'capture_origin', 'data_origin', 'generator_version', 'random_seed', 'scenario_id', 'maintenance_overdue_days_pre_service', 'maintenance_overdue_km_pre_service', 'overdue_maintenance_days_at_failure', 'overdue_maintenance_km_at_failure', 'previous_failure_count', 'failure_hazard_score', 'primary_fault_component', 'failure_severity']

dtypes:
service_id                               object
wo


Toplam missing value: 431063
Duplicate row sayısı: 0

head():


,service_id,workshop_id,customer_id,motorcycle_id,appointment_id,service_type_code,primary_trigger_task,trigger_due_km,trigger_due_date,service_delay_days,arrival_mode,received_at,odometer_km,mileage_source,mileage_quality_flag,is_mileage_estimated,status,estimated_delivery_at,completed_at,delivered_at,customer_complaint,technician_notes,labor_minutes_pre_task,labor_total,parts_total,discount_total,tax_total,grand_total,is_breakdown,is_warranty,capture_origin,data_origin,generator_version,random_seed,scenario_id,maintenance_overdue_days_pre_service,maintenance_overdue_km_pre_service,overdue_maintenance_days_at_failure,overdue_maintenance_km_at_failure,previous_failure_count,failure_hazard_score,primary_fault_component,failure_severity
0,SVC000001,WS0003,C000001,MC000001,NaN,PERIODIC,GENERAL_SAFETY_INSPECTION,NaN,NaN,0.0,WALK_IN,2024-07-03 16:15:00,84478,MANUAL,VALID,0,DELIVERED,2024-07-03 16:15:00,2024-07-03 16:56:00,2024-07-04 16:08:00,Mekanik sorun şüphesiyle genel kontrol istendi.,NaN,150,NaN,NaN,NaN,NaN,NaN,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.064147,NaN,NaN
1,SVC000002,WS0003,C000001,MC000001,NaN,PERIODIC,ENGINE_OIL_CHANGE,105139.4,2026-05-15,2.0,WALK_IN,2025-08-11 09:26:00,105231,MANUAL,VALID,0,DELIVERED,2025-08-11 09:26:00,2025-08-11 17:11:00,2025-08-11 14:39:00,Genel bakım ve kontroller talep edildi.,NaN,56,NaN,NaN,NaN,NaN,NaN,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.078228,NaN,NaN
2,SVC000003,WS0003,C000001,MC000001,APT000001,PERIODIC,ENGINE_OIL_CHANGE,115760.1,2026-11-24,28.0,APPOINTMENT,2026-03-25 11:19:00,117670,ESTIMATED,ESTIMATED,1,DELIVERED,2026-03-25 11:19:00,2026-03-25 09:12:00,2026-03-26 09:36:00,Periyodik bakım zamanı geldi.,NaN,66,NaN,NaN,NaN,NaN,NaN,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,2.0,91.6,NaN,NaN,0,0.077645,NaN,NaN
3,SVC000004,WS0003,C000001,MC000001,APT000002,REPAIR,FAULT_EVENT,122170.2,2027-03-25,NaN,APPOINTMENT,2026-06-22 15:20:00,122774,MANUAL,VALID,0,DELIVERED,2026-06-24 15:20:00,2026-06-24 12:11:00,2026-06-24 15:01:00,Bakım uyarısı sonrası servis randevusu oluşturuldu.,NaN,135,NaN,NaN,NaN,NaN,NaN,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,28.0,1909.9,28.0,1909.9,0,0.095035,TRANSMISSION,LOW
4,SVC000005,WS0008,C000002,MC000002,APT000003,PERIODIC,GENERAL_SAFETY_INSPECTION,NaN,NaN,0.0,APPOINTMENT,2023-09-19 12:40:00,30196,MANUAL,VALID,0,DELIVERED,2023-09-19 12:40:00,2023-09-19 13:22:00,2023-09-19 16:01:00,Kullanıcı arıza kontrolü talep etti.,NaN,296,NaN,NaN,NaN,NaN,NaN,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.034875,NaN,NaN


### services_enriched.csv

In [19]:
profile_dataframe(df_services_enriched, "services_enriched.csv")

=== services_enriched.csv ===
Shape: (41518, 49)

Sütunlar (tümü, 49 adet):
['service_id', 'subtotal_before_discount', 'discount_rate', 'tax_rate', 'pricing_currency', 'cost_calculation_basis', 'cost_rule_version', 'workshop_id', 'customer_id', 'motorcycle_id', 'appointment_id', 'service_type_code', 'primary_trigger_task', 'trigger_due_km', 'trigger_due_date', 'service_delay_days', 'arrival_mode', 'received_at', 'odometer_km', 'mileage_source', 'mileage_quality_flag', 'is_mileage_estimated', 'status', 'estimated_delivery_at', 'completed_at', 'delivered_at', 'customer_complaint', 'technician_notes', 'labor_minutes_pre_task', 'labor_total', 'parts_total', 'discount_total', 'tax_total', 'grand_total', 'is_breakdown', 'is_warranty', 'capture_origin', 'data_origin', 'generator_version', 'random_seed', 'scenario_id', 'maintenance_overdue_days_pre_service', 'maintenance_overdue_km_pre_service', 'overdue_maintenance_days_at_failure', 'overdue_maintenance_km_at_failure', 'previous_failure_count

Duplicate row sayısı: 0

head():


,service_id,subtotal_before_discount,discount_rate,tax_rate,pricing_currency,cost_calculation_basis,cost_rule_version,workshop_id,customer_id,motorcycle_id,appointment_id,service_type_code,primary_trigger_task,trigger_due_km,trigger_due_date,service_delay_days,arrival_mode,received_at,odometer_km,mileage_source,mileage_quality_flag,is_mileage_estimated,status,estimated_delivery_at,completed_at,delivered_at,customer_complaint,technician_notes,labor_minutes_pre_task,labor_total,parts_total,discount_total,tax_total,grand_total,is_breakdown,is_warranty,capture_origin,data_origin,generator_version,random_seed,scenario_id,maintenance_overdue_days_pre_service,maintenance_overdue_km_pre_service,overdue_maintenance_days_at_failure,overdue_maintenance_km_at_failure,previous_failure_count,failure_hazard_score,primary_fault_component,failure_severity
0,SVC000001,2830.0,0.00,0.2,TRY,SYNTHETIC_SERVICE_TASKS_X_PART_PRICES,1.2.0,WS0003,C000001,MC000001,NaN,PERIODIC,GENERAL_SAFETY_INSPECTION,NaN,NaN,0.0,WALK_IN,2024-07-03 16:15:00,84478,MANUAL,VALID,0,DELIVERED,2024-07-03 16:15:00,2024-07-03 16:56:00,2024-07-04 16:08:00,Mekanik sorun şüphesiyle genel kontrol istendi.,NaN,150,730.0,2100.0,0.00,566.00,3396.0,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.064147,NaN,NaN
1,SVC000002,2010.0,0.00,0.2,TRY,SYNTHETIC_SERVICE_TASKS_X_PART_PRICES,1.2.0,WS0003,C000001,MC000001,NaN,PERIODIC,ENGINE_OIL_CHANGE,105139.4,2026-05-15,2.0,WALK_IN,2025-08-11 09:26:00,105231,MANUAL,VALID,0,DELIVERED,2025-08-11 09:26:00,2025-08-11 17:11:00,2025-08-11 14:39:00,Genel bakım ve kontroller talep edildi.,NaN,56,1070.0,940.0,0.00,402.00,2412.0,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.078228,NaN,NaN
2,SVC000003,2450.0,0.00,0.2,TRY,SYNTHETIC_SERVICE_TASKS_X_PART_PRICES,1.2.0,WS0003,C000001,MC000001,APT000001,PERIODIC,ENGINE_OIL_CHANGE,115760.1,2026-11-24,28.0,APPOINTMENT,2026-03-25 11:19:00,117670,ESTIMATED,ESTIMATED,1,DELIVERED,2026-03-25 11:19:00,2026-03-25 09:12:00,2026-03-26 09:36:00,Periyodik bakım zamanı geldi.,NaN,66,1880.0,570.0,0.00,490.00,2940.0,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,2.0,91.6,NaN,NaN,0,0.077645,NaN,NaN
3,SVC000004,1310.0,0.00,0.2,TRY,SYNTHETIC_SERVICE_TASKS_X_PART_PRICES,1.2.0,WS0003,C000001,MC000001,APT000002,REPAIR,FAULT_EVENT,122170.2,2027-03-25,NaN,APPOINTMENT,2026-06-22 15:20:00,122774,MANUAL,VALID,0,DELIVERED,2026-06-24 15:20:00,2026-06-24 12:11:00,2026-06-24 15:01:00,Bakım uyarısı sonrası servis randevusu oluşturuldu.,NaN,135,1310.0,0.0,0.00,262.00,1572.0,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,28.0,1909.9,28.0,1909.9,0,0.095035,TRANSMISSION,LOW
4,SVC000005,2015.0,0.05,0.2,TRY,SYNTHETIC_SERVICE_TASKS_X_PART_PRICES,1.2.0,WS0008,C000002,MC000002,APT000003,PERIODIC,GENERAL_SAFETY_INSPECTION,NaN,NaN,0.0,APPOINTMENT,2023-09-19 12:40:00,30196,MANUAL,VALID,0,DELIVERED,2023-09-19 12:40:00,2023-09-19 13:22:00,2023-09-19 16:01:00,Kullanıcı arıza kontrolü talep etti.,NaN,296,1390.0,625.0,100.75,382.85,2297.1,0,0,RIDEBASE_WORKSHOP,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2,0.0,0.0,NaN,NaN,0,0.034875,NaN,NaN


### service_tasks.csv

In [20]:
profile_dataframe(df_service_tasks, "service_tasks.csv")

=== service_tasks.csv ===
Shape: (204000, 27)

Sütunlar (tümü, 27 adet):
['service_task_id', 'service_id', 'motorcycle_id', 'workshop_id', 'task_sequence', 'task_code', 'display_title', 'title_variant_type', 'component_group', 'action_type', 'trigger_reason', 'is_policy_due', 'policy_id', 'status', 'completed', 'started_at', 'completed_at', 'completed_by', 'labor_minutes', 'labor_cost', 'labor_cost_basis', 'severity', 'data_origin', 'generator_version', 'random_seed', 'notes', 'task_role']

dtypes:
service_task_id        object
service_id             object
motorcycle_id          object
workshop_id            object
task_sequence           int64
task_code              object
display_title          object
title_variant_type     object
component_group        object
action_type            object
trigger_reason         object
is_policy_due           int64
policy_id              object
status                 object
completed               int64
started_at             object
completed_at    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204000 entries, 0 to 203999
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   service_task_id     204000 non-null  object 
 1   service_id          204000 non-null  object 
 2   motorcycle_id       204000 non-null  object 
 3   workshop_id         204000 non-null  object 
 4   task_sequence       204000 non-null  int64  
 5   task_code           204000 non-null  object 
 6   display_title       204000 non-null  object 
 7   title_variant_type  204000 non-null  object 
 8   component_group     204000 non-null  object 
 9   action_type         204000 non-null  object 
 10  trigger_reason      204000 non-null  object 
 11  is_policy_due       204000 non-null  int64  
 12  policy_id           195734 non-null  object 
 13  status              204000 non-null  object 
 14  completed           204000 non-null  int64  
 15  started_at          204000 non-nul


Toplam missing value: 212896


Duplicate row sayısı: 0

head():


,service_task_id,service_id,motorcycle_id,workshop_id,task_sequence,task_code,display_title,title_variant_type,component_group,action_type,trigger_reason,is_policy_due,policy_id,status,completed,started_at,completed_at,completed_by,labor_minutes,labor_cost,labor_cost_basis,severity,data_origin,generator_version,random_seed,notes,task_role
0,STK0000001,SVC000001,MC000001,WS0003,1,BRAKE_DISC_CHANGE,Fren Diski Değişimi Yapıldı,NATURAL_VARIANT,BRAKES,REPLACE,INSPECTION_FINDING,0,NaN,COMPLETED,1,2024-07-03 16:39:02,2024-07-03 16:56:00,TECH-WS0003-03,43,420.0,SYNTHETIC_YEAR_X_WORKSHOP_RATE,MEDIUM,SYNTHETIC,1.2.0,42,NaN,PERIODIC
1,STK0000002,SVC000001,MC000001,WS0003,2,CHARGING_SYSTEM_TEST,Şarj Sistemi Test Edildi,NATURAL_VARIANT,ELECTRICAL,DIAGNOSE,FAULT,0,NaN,COMPLETED,1,2024-07-03 16:17:23,2024-07-03 16:48:23,TECH-WS0003-08,31,310.0,SYNTHETIC_YEAR_X_WORKSHOP_RATE,MEDIUM,SYNTHETIC,1.2.0,42,NaN,PERIODIC
2,STK0000003,SVC000002,MC000001,WS0003,1,ENGINE_OIL_CHANGE,Motor Yağı Değişimi,CANONICAL,ENGINE,REPLACE,MILEAGE_AND_TIME,1,POL0071,COMPLETED,1,2025-08-11 16:38:15,2025-08-11 16:58:15,TECH-WS0003-05,20,260.0,SYNTHETIC_YEAR_X_WORKSHOP_RATE,NORMAL,SYNTHETIC,1.2.0,42,NaN,PERIODIC
3,STK0000004,SVC000002,MC000001,WS0003,2,AIR_FILTER_CHANGE,Hava Filtresi Değişimi Yapıldı,NATURAL_VARIANT,INTAKE,REPLACE,MILEAGE,1,POL0073,COMPLETED,1,2025-08-11 15:57:26,2025-08-11 16:19:26,TECH-WS0003-04,22,280.0,SYNTHETIC_YEAR_X_WORKSHOP_RATE,NORMAL,SYNTHETIC,1.2.0,42,NaN,PERIODIC
4,STK0000005,SVC000002,MC000001,WS0003,3,CVT_CASE_INSPECTION,CVT Kutu Kontrolü ve genel durum değerlendirmesi,NATURAL_VARIANT,TRANSMISSION,INSPECT,MILEAGE,1,POL0095,COMPLETED,1,2025-08-11 14:14:58,2025-08-11 14:28:58,TECH-WS0003-07,14,180.0,SYNTHETIC_YEAR_X_WORKSHOP_RATE,NORMAL,SYNTHETIC,1.2.0,42,NaN,PERIODIC


### service_parts.csv

In [21]:
profile_dataframe(df_service_parts, "service_parts.csv")

=== service_parts.csv ===
Shape: (76305, 27)

Sütunlar (tümü, 27 adet):
['service_part_id', 'service_id', 'service_task_id', 'motorcycle_id', 'workshop_id', 'task_code', 'part_id', 'part_category', 'compatibility_id', 'compatibility_level', 'quantity', 'quantity_unit', 'quantity_source', 'quantity_confidence', 'part_price_id', 'unit_purchase_price', 'unit_sale_price', 'purchase_total', 'sale_total', 'currency', 'stock_source', 'is_compatible', 'price_basis', 'data_origin', 'generator_version', 'random_seed', 'notes']

dtypes:
service_part_id         object
service_id              object
service_task_id         object
motorcycle_id           object
workshop_id             object
task_code               object
part_id                 object
part_category           object
compatibility_id        object
compatibility_level     object
quantity               float64
quantity_unit           object
quantity_source         object
quantity_confidence     object
part_price_id           object
uni


Toplam missing value: 0
Duplicate row sayısı: 0

head():


,service_part_id,service_id,service_task_id,motorcycle_id,workshop_id,task_code,part_id,part_category,compatibility_id,compatibility_level,quantity,quantity_unit,quantity_source,quantity_confidence,part_price_id,unit_purchase_price,unit_sale_price,purchase_total,sale_total,currency,stock_source,is_compatible,price_basis,data_origin,generator_version,random_seed,notes
0,SPT0000001,SVC000001,STK0000001,MC000001,WS0003,BRAKE_DISC_CHANGE,PRT0499,BRAKE_DISC,CMP00704,SYNTHETIC_MODEL_REPAIR_FITMENT,1.0,PIECE,TASK_DEFAULT,MEDIUM,PPR0029896,1400.0,2100.0,1400.0,2100.0,TRY,WORKSHOP_STOCK,1,SYNTHETIC_SIMULATION,SYNTHETIC,1.2.0,42,Selected using compatibility precedence and service-year/workshop price history.
1,SPT0000002,SVC000002,STK0000003,MC000001,WS0003,ENGINE_OIL_CHANGE,PRT0011,ENGINE_OIL,CMP00143,SYNTHETIC_GROUP_BACKOFF,1.0,LITER,SIMULATION_DISPLACEMENT_BACKOFF,LOW,PPR0000617,260.0,340.0,260.0,340.0,TRY,WORKSHOP_STOCK,1,SYNTHETIC_SIMULATION,SYNTHETIC,1.2.0,42,Selected using compatibility precedence and service-year/workshop price history.
2,SPT0000003,SVC000002,STK0000004,MC000001,WS0003,AIR_FILTER_CHANGE,PRT0118,AIR_FILTER,CMP00144,SYNTHETIC_MODEL_SPECIFIC,1.0,PIECE,TASK_DEFAULT,MEDIUM,PPR0007037,390.0,600.0,390.0,600.0,TRY,WORKSHOP_STOCK,1,SYNTHETIC_SIMULATION,SYNTHETIC,1.2.0,42,Selected using compatibility precedence and service-year/workshop price history.
3,SPT0000004,SVC000003,STK0000007,MC000001,WS0003,ENGINE_OIL_CHANGE,PRT0011,ENGINE_OIL,CMP00143,SYNTHETIC_GROUP_BACKOFF,1.0,LITER,SIMULATION_DISPLACEMENT_BACKOFF,LOW,PPR0000618,280.0,370.0,280.0,370.0,TRY,ORDERED_SUPPLIER,1,SYNTHETIC_SIMULATION,SYNTHETIC,1.2.0,42,Selected using compatibility precedence and service-year/workshop price history.
4,SPT0000005,SVC000003,STK0000011,MC000001,WS0003,COOLANT_CHANGE,PRT0005,COOLANT,CMP00153,UNIVERSAL_SERVICE_CONSUMABLE,0.8,LITER,SIMULATION_DISPLACEMENT_BACKOFF,LOW,PPR0000258,180.0,250.0,144.0,200.0,TRY,WORKSHOP_STOCK,1,SYNTHETIC_SIMULATION,SYNTHETIC,1.2.0,42,Selected using compatibility precedence and service-year/workshop price history.


### service_status_history.csv

In [22]:
profile_dataframe(df_service_status_history, "service_status_history.csv")

=== service_status_history.csv ===
Shape: (312644, 21)

Sütunlar (tümü, 21 adet):
['status_history_id', 'service_id', 'workshop_id', 'customer_id', 'motorcycle_id', 'status_sequence', 'previous_status', 'status', 'status_at', 'status_reason_code', 'related_entity_type', 'related_entity_id', 'timestamp_source', 'is_derived_timestamp', 'was_timestamp_repaired', 'repair_reason', 'is_terminal', 'data_origin', 'generator_version', 'random_seed', 'scenario_id']

dtypes:
status_history_id         object
service_id                object
workshop_id               object
customer_id               object
motorcycle_id             object
status_sequence            int64
previous_status           object
status                    object
status_at                 object
status_reason_code        object
related_entity_type       object
related_entity_id         object
timestamp_source          object
is_derived_timestamp       int64
was_timestamp_repaired     int64
repair_reason             object
is_

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312644 entries, 0 to 312643
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   status_history_id       312644 non-null  object
 1   service_id              312644 non-null  object
 2   workshop_id             312644 non-null  object
 3   customer_id             312644 non-null  object
 4   motorcycle_id           312644 non-null  object
 5   status_sequence         312644 non-null  int64 
 6   previous_status         271126 non-null  object
 7   status                  312644 non-null  object
 8   status_at               312644 non-null  object
 9   status_reason_code      312644 non-null  object
 10  related_entity_type     312644 non-null  object
 11  related_entity_id       312644 non-null  object
 12  timestamp_source        312644 non-null  object
 13  is_derived_timestamp    312644 non-null  int64 
 14  was_timestamp_repaired  312644 non-n


Toplam missing value: 313253


Duplicate row sayısı: 0

head():


,status_history_id,service_id,workshop_id,customer_id,motorcycle_id,status_sequence,previous_status,status,status_at,status_reason_code,related_entity_type,related_entity_id,timestamp_source,is_derived_timestamp,was_timestamp_repaired,repair_reason,is_terminal,data_origin,generator_version,random_seed,scenario_id
0,SSH0000001,SVC000001,WS0003,C000001,MC000001,1,NaN,RECEIVED,2024-07-03 16:15:00,SERVICE_ACCEPTED,SERVICE,SVC000001,SERVICES.RECEIVED_AT,0,0,NaN,0,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2
1,SSH0000002,SVC000001,WS0003,C000001,MC000001,2,RECEIVED,DIAGNOSIS,2024-07-03 16:15:28,INITIAL_DIAGNOSIS,SERVICE,SVC000001,DERIVED_RECEIVED_TO_FIRST_TASK,1,0,NaN,0,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2
2,SSH0000003,SVC000001,WS0003,C000001,MC000001,3,DIAGNOSIS,AWAITING_APPROVAL,2024-07-03 16:16:04,CUSTOMER_APPROVAL_FLOW,SERVICE,SVC000001,DERIVED_RECEIVED_TO_FIRST_TASK,1,0,NaN,0,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2
3,SSH0000004,SVC000001,WS0003,C000001,MC000001,4,AWAITING_APPROVAL,PARTS_RESERVED,2024-07-03 16:17:08,PARTS_ALLOCATED,SERVICE_PART,SPT0000001,DERIVED_FROM_SERVICE_PARTS,1,0,NaN,0,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2
4,SSH0000005,SVC000001,WS0003,C000001,MC000001,5,PARTS_RESERVED,IN_PROGRESS,2024-07-03 16:17:23,FIRST_TASK_STARTED,SERVICE_TASK,STK0000002,SERVICE_TASKS.STARTED_AT,0,0,NaN,0,SYNTHETIC,1.2.0,42,HAZARD_EVENT_SIM_V1_2


### noise_audit.csv

In [23]:
profile_dataframe(df_noise_audit, "noise_audit.csv")

=== noise_audit.csv ===
Shape: (46621, 26)

Sütunlar (tümü, 26 adet):
['noise_audit_id', 'entity_type', 'entity_id', 'service_id', 'motorcycle_id', 'workshop_id', 'field_name', 'issue_type', 'issue_category', 'severity', 'is_intentional_noise', 'observed_value', 'original_clean_value', 'canonical_clean_value', 'clean_value_available', 'detection_source', 'repair_action', 'repair_applied', 'ml_handling', 'noise_rule_version', 'data_origin', 'generator_version', 'random_seed', 'scenario_id', 'notes', 'original_clean_value_available']

dtypes:
noise_audit_id                     object
entity_type                        object
entity_id                          object
service_id                         object
motorcycle_id                      object
workshop_id                        object
field_name                         object
issue_type                         object
issue_category                     object
severity                           object
is_intentional_noise             


Toplam missing value: 47403
Duplicate row sayısı: 0

head():


,noise_audit_id,entity_type,entity_id,service_id,motorcycle_id,workshop_id,field_name,issue_type,issue_category,severity,is_intentional_noise,observed_value,original_clean_value,canonical_clean_value,clean_value_available,detection_source,repair_action,repair_applied,ml_handling,noise_rule_version,data_origin,generator_version,random_seed,scenario_id,notes,original_clean_value_available
0,NAU0000001,SERVICE_STATUS,SSH0000014,SVC000002,MC000001,WS0003,delivered_at,STATUS_TIMESTAMP_INCONSISTENCY,TEMPORAL_INCONSISTENCY,HIGH,0,2025-08-11 14:39:00,NaN,2025-08-11 17:12:00,1,REPAIRED_SERVICES.DELIVERED_AT,USE_CANONICAL_STATUS_HISTORY_TIMESTAMP,1,USE_REPAIRED_TIMELINE_ONLY,STATUS_TIMELINE_REPAIR_V1_1,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,DELIVERY_NOT_AFTER_COMPLETION,0
1,NAU0000002,SERVICE,SVC000003,SVC000003,MC000001,WS0003,odometer_km,MILEAGE_ESTIMATED,OBSERVATION_UNCERTAINTY,MEDIUM,0,117670,NaN,NaN,0,services.is_mileage_estimated,KEEP_WITH_QUALITY_FLAG,0,KEEP_AND_INCLUDE_MILEAGE_QUALITY_FEATURES,MILEAGE_UNCERTAINTY_V1_1,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,This is modeled measurement uncertainty rather than a known corrupted value. No hidden true odometer value is claimed by the audit table.,0
2,NAU0000003,SERVICE_STATUS,SSH0000021,SVC000003,MC000001,WS0003,completed_at,STATUS_TIMESTAMP_INCONSISTENCY,TEMPORAL_INCONSISTENCY,HIGH,0,2026-03-25 09:12:00,NaN,2026-03-25 11:19:00,1,SERVICE_TASKS.COMPLETED_AT,USE_CANONICAL_STATUS_HISTORY_TIMESTAMP,1,USE_REPAIRED_TIMELINE_ONLY,STATUS_TIMELINE_REPAIR_V1_1,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,SERVICE_COMPLETED_BEFORE_LAST_TASK_COMPLETION,0
3,NAU0000004,SERVICE_STATUS,SSH0000042,SVC000006,MC000002,WS0008,completed_at,STATUS_TIMESTAMP_INCONSISTENCY,TEMPORAL_INCONSISTENCY,HIGH,0,2023-11-13 11:11:00,NaN,2023-11-13 14:24:00,1,SERVICE_TASKS.COMPLETED_AT,USE_CANONICAL_STATUS_HISTORY_TIMESTAMP,1,USE_REPAIRED_TIMELINE_ONLY,STATUS_TIMELINE_REPAIR_V1_1,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,SERVICE_COMPLETED_BEFORE_LAST_TASK_COMPLETION,0
4,NAU0000005,SERVICE_STATUS,SSH0000043,SVC000006,MC000002,WS0008,delivered_at,STATUS_TIMESTAMP_INCONSISTENCY,TEMPORAL_INCONSISTENCY,HIGH,0,2023-11-13 09:39:00,NaN,2023-11-13 14:25:00,1,REPAIRED_SERVICES.DELIVERED_AT,USE_CANONICAL_STATUS_HISTORY_TIMESTAMP,1,USE_REPAIRED_TIMELINE_ONLY,STATUS_TIMELINE_REPAIR_V1_1,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,DELIVERY_NOT_AFTER_COMPLETION,0


### ml_maintenance_snapshots.parquet

In [24]:
profile_dataframe(df_ml_snapshots, "ml_maintenance_snapshots.parquet")

=== ml_maintenance_snapshots.parquet ===
Shape: (41518, 142)

Sütunlar (tümü, 142 adet):
['snapshot_id', 'source_service_id', 'snapshot_at', 'snapshot_date', 'service_sequence', 'snapshot_year', 'snapshot_month', 'snapshot_quarter', 'snapshot_day_of_year', 'snapshot_month_sin', 'snapshot_month_cos', 'motorcycle_id', 'customer_id', 'workshop_id', 'model_id', 'brand', 'model_name', 'category', 'powertrain_type', 'engine_displacement_cc', 'cylinder_count', 'cooling_type', 'final_drive_type', 'transmission_type', 'engine_oil_service_qty_l', 'spark_plug_count', 'fuel_type', 'policy_group', 'policy_ready', 'spec_confidence', 'production_year', 'motorcycle_age_years', 'ownership_months', 'initial_mileage_km', 'is_left_truncated', 'days_observed', 'customer_type', 'is_fleet_customer', 'acquisition_channel', 'usage_type', 'annual_km_baseline', 'city_ratio', 'highway_ratio', 'offroad_ratio', 'track_ratio', 'avg_ride_days_per_week', 'daily_active_probability', 'avg_km_per_active_day', 'riding_int

Duplicate row sayısı: 0

head():


,snapshot_id,source_service_id,snapshot_at,snapshot_date,service_sequence,snapshot_year,snapshot_month,snapshot_quarter,snapshot_day_of_year,snapshot_month_sin,snapshot_month_cos,motorcycle_id,customer_id,workshop_id,model_id,brand,model_name,category,powertrain_type,engine_displacement_cc,cylinder_count,cooling_type,final_drive_type,transmission_type,engine_oil_service_qty_l,spark_plug_count,fuel_type,policy_group,policy_ready,spec_confidence,production_year,motorcycle_age_years,ownership_months,initial_mileage_km,is_left_truncated,days_observed,customer_type,is_fleet_customer,acquisition_channel,usage_type,annual_km_baseline,city_ratio,highway_ratio,offroad_ratio,track_ratio,avg_ride_days_per_week,daily_active_probability,avg_km_per_active_day,riding_intensity,seasonality_strength,winter_usage_factor,weather_sensitivity,storage_condition,load_severity_factor,climate_zone,workshop_city,workshop_region,workshop_climate_zone,price_level,service_bay_count,technician_count,daily_service_capacity,appointment_rate,avg_parts_lead_days,same_day_parts_rate,labor_rate_index,customer_volume_index,courier_customer_share,snapshot_odometer_km,current_mileage_source,current_mileage_quality_flag,current_mileage_estimated,current_service_type_code,current_arrival_mode,current_primary_trigger_task,current_service_delay_days,current_is_breakdown,current_is_warranty,current_service_timestamp_repaired,current_labor_total,current_parts_total,current_grand_total,current_task_count,current_completed_task_count,current_declined_task_count,current_policy_due_task_count,current_fault_task_count,current_inspection_finding_task_count,current_replace_task_count,current_parts_line_count,current_supplier_parts_line_count,days_since_previous_service,km_since_previous_service,avg_km_per_day_since_previous_service,avg_service_interval_days,avg_service_interval_km,rolling3_interval_days,rolling3_interval_km,service_odometer_regression_count_to_date,periodic_service_count,repair_service_count,breakdown_service_count,warranty_service_count,appointment_service_count,walkin_service_count,services_last_90d,services_last_365d,cumulative_service_spend,avg_service_spend,total_task_count,total_completed_task_count,total_declined_task_count,total_policy_due_task_count,total_fault_task_count,total_inspection_finding_task_count,total_replace_task_count,days_since_engine_task,km_since_engine_task,days_since_brakes_task,km_since_brakes_task,days_since_final_drive_task,km_since_final_drive_task,days_since_tires_wheels_task,km_since_tires_wheels_task,days_since_electrical_task,km_since_electrical_task,days_since_transmission_task,km_since_transmission_task,days_since_cooling_task,km_since_cooling_task,days_since_intake_task,km_since_intake_task,feature_version,data_origin,generator_version,random_seed,scenario_id,maintenance_overdue_days_pre_service,maintenance_overdue_km_pre_service,previous_failure_count,failure_hazard_score,chronic_risk_tier
0,SNP_SVC000001,SVC000001,2024-07-04 16:08:00,2024-07-04,1,2024,7,3,186,1.224647e-16,-1.000000,MC000001,C000001,WS0003,YAMAHA_NMAX125,Yamaha,NMAX 125,SCOOTER,ICE,125.0,1,LIQUID,V_BELT,CVT,NaN,1.0,GASOLINE,ICE_125_LIQUID_BELT,True,HIGH,2017,7.378508,65.938398,83780,1,13,INDIVIDUAL,0,GOOGLE_MAPS,COMMUTER,18507,0.949,0.0462,0.0048,0.0,5.70,0.77,71.4,HIGH,0.145,0.811,0.264,COVERED_PARKING,1.032,MEDITERRANEAN_COASTAL,İzmir,Ege,MEDITERRANEAN_COASTAL,MEDIUM,7,8,18,0.62,1.5,0.82,1.02,1.18,0.22,84478,MANUAL,VALID,0,REPAIR,WALK_IN,UNSCHEDULED_EVENT,0,0,0,0,730.0,2100.0,3396.0,2,2,0,0,2,1,1,1,0,-1.000000,-1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0,1,0,0,0,0,1,1,1,3396.0,3396.0,2,2,0,0,2,1,1,-1.000000,-1.0,0.000000,0.0,-1.0,-1.0,-1.0,-1.0,0.000000,0.0,-1.000000,-1.0,-1.000000,-1.0,-1.000000,-1.0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,0.0,0.0,0,0.064147,NORMAL
1,SNP_SVC000002,SVC000002,2025-08-11 17:12:00,2025-08-11,2,2025,8,3,223,-5.000000e-01,-0.866025,MC000001,C000001,WS0003,YAMAHA_NMAX125,Yamaha,NMAX 125,SCOOTER,ICE,125.0

### ml_next_service_targets.parquet

In [25]:
profile_dataframe(df_next_service_targets, "ml_next_service_targets.parquet")

=== ml_next_service_targets.parquet ===
Shape: (41518, 38)

Sütunlar (tümü, 38 adet):
['snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'snapshot_at', 'snapshot_odometer_km', 'target_event_observed', 'is_right_censored', 'next_service_id', 'next_service_received_at_raw', 'target_next_service_at', 'next_service_delivered_at', 'days_to_next_service_raw', 'days_to_next_service', 'target_time_repaired', 'next_service_odometer_km', 'km_to_next_service_raw', 'km_to_next_service', 'target_km_valid', 'next_service_mileage_source', 'next_service_mileage_quality_flag', 'next_service_is_mileage_estimated', 'next_service_type_code', 'next_service_primary_trigger_task', 'next_service_arrival_mode', 'next_service_is_breakdown', 'next_service_is_warranty', 'censor_at', 'censor_days', 'days_to_event_or_censor', 'censor_source', 'censor_time_repaired', 'target_version', 'data_origin', 'generator_version', 'random_seed', 'scenario_id']

dtypes:
snapshot_id              

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41518 entries, 0 to 41517
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   snapshot_id                        41518 non-null  object 
 1   source_service_id                  41518 non-null  object 
 2   motorcycle_id                      41518 non-null  object 
 3   customer_id                        41518 non-null  object 
 4   workshop_id                        41518 non-null  object 
 5   snapshot_at                        41518 non-null  object 
 6   snapshot_odometer_km               41518 non-null  int64  
 7   target_event_observed              41518 non-null  int32  
 8   is_right_censored                  41518 non-null  int32  
 9   next_service_id                    33076 non-null  object 
 10  next_service_received_at_raw       33076 non-null  object 
 11  target_next_service_at             33076 non-null  obj

,snapshot_id,source_service_id,motorcycle_id,customer_id,workshop_id,snapshot_at,snapshot_odometer_km,target_event_observed,is_right_censored,next_service_id,next_service_received_at_raw,target_next_service_at,next_service_delivered_at,days_to_next_service_raw,days_to_next_service,target_time_repaired,next_service_odometer_km,km_to_next_service_raw,km_to_next_service,target_km_valid,next_service_mileage_source,next_service_mileage_quality_flag,next_service_is_mileage_estimated,next_service_type_code,next_service_primary_trigger_task,next_service_arrival_mode,next_service_is_breakdown,next_service_is_warranty,censor_at,censor_days,days_to_event_or_censor,censor_source,censor_time_repaired,target_version,data_origin,generator_version,random_seed,scenario_id
0,SNP_SVC000001,SVC000001,MC000001,C000001,WS0003,2024-07-04 16:08:00,84478,1,0,SVC000002,2025-08-11 09:26:00,2025-08-11 09:26:00,2025-08-11 17:12:00,402.720833,402.720833,0.0,105231.0,20753.0,20753.0,1.0,MANUAL,VALID,0.0,PERIODIC,ENGINE_OIL_CHANGE,WALK_IN,0.0,0.0,2026-08-03 17:33:00,760.059028,402.720833,GLOBAL_DATASET_CUTOFF,0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1
1,SNP_SVC000002,SVC000002,MC000001,C000001,WS0003,2025-08-11 17:12:00,105231,1,0,SVC000003,2026-03-25 11:19:00,2026-03-25 11:19:00,2026-03-26 09:36:00,225.754861,225.754861,0.0,117670.0,12439.0,12439.0,1.0,ESTIMATED,ESTIMATED,1.0,PERIODIC,ENGINE_OIL_CHANGE,APPOINTMENT,0.0,0.0,2026-08-03 17:33:00,357.014583,225.754861,GLOBAL_DATASET_CUTOFF,0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1
2,SNP_SVC000003,SVC000003,MC000001,C000001,WS0003,2026-03-26 09:36:00,117670,1,0,SVC000004,2026-06-22 15:20:00,2026-06-22 15:20:00,2026-06-24 15:01:00,88.238889,88.238889,0.0,122774.0,5104.0,5104.0,1.0,MANUAL,VALID,0.0,REPAIR,FAULT_EVENT,APPOINTMENT,0.0,0.0,2026-08-03 17:33:00,130.331250,88.238889,GLOBAL_DATASET_CUTOFF,0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1
3,SNP_SVC000004,SVC000004,MC000001,C000001,WS0003,2026-06-24 15:01:00,122774,0,1,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,NaN,2026-08-03 17:33:00,40.105556,40.105556,GLOBAL_DATASET_CUTOFF,0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1
4,SNP_SVC000005,SVC000005,MC000002,C000002,WS0008,2023-09-19 16:01:00,30196,1,0,SVC000006,2023-11-13 14:24:00,2023-11-13 14:24:00,2023-11-13 14:25:00,54.932639,54.932639,0.0,32227.0,2031.0,2031.0,1.0,MANUAL,VALID,0.0,PERIODIC,ENGINE_OIL_CHANGE,WALK_IN,0.0,0.0,2026-08-03 17:33:00,1049.063889,54.932639,GLOBAL_DATASET_CUTOFF,0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1


### ml_next_task_targets.parquet

In [26]:
profile_dataframe(df_next_task_targets, "ml_next_task_targets.parquet")

=== ml_next_task_targets.parquet ===
Shape: (41518, 122)

Sütunlar (tümü, 122 adet):
['snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'next_service_id', 'target_event_observed', 'is_right_censored', 'task__ENGINE_OIL_CHECK', 'task__ENGINE_OIL_CHANGE', 'task__OIL_FILTER_CHANGE', 'task__OIL_LEAK_INSPECTION', 'task__ENGINE_COMPRESSION_TEST', 'task__VALVE_CLEARANCE_INSPECTION', 'task__VALVE_CLEARANCE_ADJUST', 'task__CAM_CHAIN_INSPECTION', 'task__CAM_CHAIN_TENSIONER_SERVICE', 'task__AIR_FILTER_INSPECTION', 'task__AIR_FILTER_CLEAN', 'task__AIR_FILTER_CHANGE', 'task__THROTTLE_BODY_CLEAN', 'task__INTAKE_LEAK_INSPECTION', 'task__SPARK_PLUG_INSPECTION', 'task__SPARK_PLUG_CHANGE', 'task__FUEL_FILTER_INSPECTION', 'task__FUEL_FILTER_CHANGE', 'task__FUEL_INJECTOR_CLEAN', 'task__FUEL_SYSTEM_DIAGNOSTIC', 'task__COOLANT_LEVEL_CHECK', 'task__COOLANT_CHANGE', 'task__COOLING_SYSTEM_INSPECTION', 'task__RADIATOR_CLEAN', 'task__THERMOSTAT_DIAGNOSTIC', 'task__CLUTCH_FREE_PLA


Toplam missing value: 918351
Duplicate row sayısı: 0

head():


,snapshot_id,source_service_id,motorcycle_id,customer_id,workshop_id,next_service_id,target_event_observed,is_right_censored,task__ENGINE_OIL_CHECK,task__ENGINE_OIL_CHANGE,task__OIL_FILTER_CHANGE,task__OIL_LEAK_INSPECTION,task__ENGINE_COMPRESSION_TEST,task__VALVE_CLEARANCE_INSPECTION,task__VALVE_CLEARANCE_ADJUST,task__CAM_CHAIN_INSPECTION,task__CAM_CHAIN_TENSIONER_SERVICE,task__AIR_FILTER_INSPECTION,task__AIR_FILTER_CLEAN,task__AIR_FILTER_CHANGE,task__THROTTLE_BODY_CLEAN,task__INTAKE_LEAK_INSPECTION,task__SPARK_PLUG_INSPECTION,task__SPARK_PLUG_CHANGE,task__FUEL_FILTER_INSPECTION,task__FUEL_FILTER_CHANGE,task__FUEL_INJECTOR_CLEAN,task__FUEL_SYSTEM_DIAGNOSTIC,task__COOLANT_LEVEL_CHECK,task__COOLANT_CHANGE,task__COOLING_SYSTEM_INSPECTION,task__RADIATOR_CLEAN,task__THERMOSTAT_DIAGNOSTIC,task__CLUTCH_FREE_PLAY_CHECK,task__CLUTCH_ADJUST,task__CLUTCH_PLATE_INSPECTION,task__CLUTCH_PLATE_CHANGE,task__GEARBOX_OIL_CHANGE,task__CVT_CASE_INSPECTION,task__CVT_BELT_INSPECTION,task__CVT_BELT_CHANGE,task__CVT_ROLLER_INSPECTION,task__CVT_ROLLER_CHANGE,task__FINAL_GEAR_OIL_CHANGE,task__CHAIN_INSPECTION,task__CHAIN_CLEAN,task__CHAIN_LUBRICATE,task__CHAIN_ADJUST,task__CHAIN_SPROCKET_REPLACE,task__DRIVE_BELT_INSPECTION,task__DRIVE_BELT_CHANGE,task__SHAFT_DRIVE_OIL_CHANGE,task__SHAFT_DRIVE_INSPECTION,task__FRONT_BRAKE_PAD_INSPECTION,task__FRONT_BRAKE_PAD_CHANGE,task__REAR_BRAKE_PAD_INSPECTION,task__REAR_BRAKE_PAD_CHANGE,task__BRAKE_DISC_INSPECTION,task__BRAKE_DISC_CHANGE,task__BRAKE_FLUID_CHECK,task__BRAKE_FLUID_CHANGE,task__BRAKE_SYSTEM_BLEED,task__ABS_DIAGNOSTIC,task__FRONT_TIRE_INSPECTION,task__FRONT_TIRE_CHANGE,task__REAR_TIRE_INSPECTION,task__REAR_TIRE_CHANGE,task__WHEEL_BALANCE,task__WHEEL_BEARING_INSPECTION,task__WHEEL_BEARING_CHANGE,task__FORK_INSPECTION,task__FORK_OIL_CHANGE,task__FORK_SEAL_CHANGE,task__REAR_SHOCK_INSPECTION,task__STEERING_BEARING_INSPECTION,task__STEERING_BEARING_ADJUST,task__STEERING_BEARING_CHANGE,task__BATTERY_TEST,task__BATTERY_CHANGE,task__CHARGING_SYSTEM_TEST,task__LIGHTING_SYSTEM_INSPECTION,task__ECU_DIAGNOSTIC_SCAN,task__EV_TRACTION_BATTERY_HEALTH_CHECK,task__EV_CHARGING_PORT_INSPECTION,task__EV_DRIVE_MOTOR_DIAGNOSTIC,task__EV_HIGH_VOLTAGE_SYSTEM_INSPECTION,task__GENERAL_SAFETY_INSPECTION,task__FASTENER_TORQUE_INSPECTION,task__CABLE_CONTROL_LUBRICATION,target_definition,taxonomy_target_class_count,observed_positive_class_count,eligible_but_unobserved_class_count,taxonomy_version,target_version,data_origin,generator_version,random_seed,scenario_id,next_task_codes,next_completed_task_codes,next_task_count,next_completed_task_count,next_declined_task_count,next_declined_task_codes,task__BRAKE_FAULT_DIAGNOSTIC,task__CENTER_STAND_INSPECTION,task__CHAIN_DRIVE_DIAGNOSTIC,task__COOLING_FAULT_DIAGNOSTIC,task__CVT_BELT_FAILURE_DIAGNOSTIC,task__ELECTRICAL_FAULT_DIAGNOSTIC,task__ENGINE_FAULT_DIAGNOSTIC,task__FINAL_DRIVE_FAULT_INSPECTION,task__INTAKE_FAULT_DIAGNOSTIC,task__SIDE_STAND_INSPECTION,task__STEERING_FAULT_INSPECTION,task__SUSPENSION_FAULT_INSPECTION,task__THROTTLE_FREE_PLAY_CHECK,task__TIRE_PRESSURE_CHECK,task__TIRE_WHEEL_FAULT_INSPECTION,task__TRANSMISSION_FAULT_DIAGNOSTIC,task__WASH_AND_GENERAL_CLEANING
0,SNP_SVC000001,SVC000001,MC000001,C000001,WS0003,SVC000002,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,TASK_PRESENT_IN_NEXT_SERVICE_PLAN_COMPLETED_OR_DECLINED,81,60,21,1.2.0,1.2.0,SYNTHETIC,1.2.0,42,MAINTENANCE_EVENT_SIM_V1,AIR_FILTER_CHANGE|CVT_CASE_INSPECTION|ENGINE_OIL_CHANGE|GENERAL_SAFETY_INSPECTION,AIR_FILTER_CHANGE|CVT_CASE_INSPECTION|ENGINE_OIL_CHANGE|GENERAL_SAFETY_INSPECTION,4,4,0,None,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,SNP_SVC000002,SVC000002,MC000001,C000001,WS0003,SVC000003,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,TASK_PRESENT_IN_NEXT_SERVICE_PLAN_COMPLETED_OR

### split_manifest.csv

In [27]:
profile_dataframe(df_split_manifest, "split_manifest.csv")

=== split_manifest.csv ===
Shape: (41518, 37)

Sütunlar (tümü, 37 adet):
['manifest_id', 'snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'snapshot_at', 'snapshot_odometer_km', 'primary_time_split', 'primary_split_start_at', 'primary_split_end_at', 'primary_label_cutoff_at', 'next_service_id', 'next_service_at', 'full_dataset_event_observed', 'full_dataset_right_censored', 'event_observed_by_primary_cutoff', 'boundary_crossing_future_target', 'task_target_eligible_primary', 'next_service_regression_eligible_primary', 'survival_target_eligible_primary', 'survival_event_observed_in_window', 'survival_admin_censor_at', 'unseen_motorcycle_split', 'is_unseen_motorcycle_validation', 'is_unseen_motorcycle_test', 'unseen_workshop_split', 'is_unseen_workshop_validation', 'is_unseen_workshop_test', 'motorcycle_group_hash', 'workshop_group_hash', 'split_version', 'random_seed', 'data_origin', 'generator_version', 'scenario_id', 'notes']

dtypes:
manifest_id      

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41518 entries, 0 to 41517
Data columns (total 37 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   manifest_id                               41518 non-null  object
 1   snapshot_id                               41518 non-null  object
 2   source_service_id                         41518 non-null  object
 3   motorcycle_id                             41518 non-null  object
 4   customer_id                               41518 non-null  object
 5   workshop_id                               41518 non-null  object
 6   snapshot_at                               41518 non-null  object
 7   snapshot_odometer_km                      41518 non-null  int64 
 8   primary_time_split                        41518 non-null  object
 9   primary_split_start_at                    41518 non-null  object
 10  primary_split_end_at                      4151

Duplicate row sayısı: 0

head():


,manifest_id,snapshot_id,source_service_id,motorcycle_id,customer_id,workshop_id,snapshot_at,snapshot_odometer_km,primary_time_split,primary_split_start_at,primary_split_end_at,primary_label_cutoff_at,next_service_id,next_service_at,full_dataset_event_observed,full_dataset_right_censored,event_observed_by_primary_cutoff,boundary_crossing_future_target,task_target_eligible_primary,next_service_regression_eligible_primary,survival_target_eligible_primary,survival_event_observed_in_window,survival_admin_censor_at,unseen_motorcycle_split,is_unseen_motorcycle_validation,is_unseen_motorcycle_test,unseen_workshop_split,is_unseen_workshop_validation,is_unseen_workshop_test,motorcycle_group_hash,workshop_group_hash,split_version,random_seed,data_origin,generator_version,scenario_id,notes
0,SPM0000001,SNP_SVC000001,SVC000001,MC000001,C000001,WS0003,2024-07-04 16:08:00,84478,TRAIN,2021-01-27 11:31:00,2025-06-30 23:59:59,2025-06-30 23:59:59,SVC000002,2025-08-11 09:26:00,1,0,0,1,0,0,1,0,2025-06-30 23:59:59,TRAIN,0,0,TRAIN,0,0,6411767099563454294,5993427844424968809,SPLIT_MANIFEST_V1_2,42,SYNTHETIC,1.2.0,MAINTENANCE_EVENT_SIM_V1,PRIMARY=time-based; unseen motorcycle/workshop are separate group-holdout regimes. Direct next-event/task labels are usable only when observed by the primary split cutoff.
1,SPM0000002,SNP_SVC000002,SVC000002,MC000001,C000001,WS0003,2025-08-11 17:12:00,105231,VALIDATION,2025-07-01 00:00:00,2025-12-31 23:59:59,2025-12-31 23:59:59,SVC000003,2026-03-25 11:19:00,1,0,0,1,0,0,1,0,2025-12-31 23:59:59,TRAIN,0,0,TRAIN,0,0,6411767099563454294,5993427844424968809,SPLIT_MANIFEST_V1_2,42,SYNTHETIC,1.2.0,MAINTENANCE_EVENT_SIM_V1,PRIMARY=time-based; unseen motorcycle/workshop are separate group-holdout regimes. Direct next-event/task labels are usable only when observed by the primary split cutoff.
2,SPM0000003,SNP_SVC000003,SVC000003,MC000001,C000001,WS0003,2026-03-26 09:36:00,117670,TEST,2026-01-01 00:00:00,2026-08-03 17:33:00,2026-08-03 17:33:00,SVC000004,2026-06-22 15:20:00,1,0,1,0,1,1,1,1,2026-06-22 15:20:00,TRAIN,0,0,TRAIN,0,0,6411767099563454294,5993427844424968809,SPLIT_MANIFEST_V1_2,42,SYNTHETIC,1.2.0,MAINTENANCE_EVENT_SIM_V1,PRIMARY=time-based; unseen motorcycle/workshop are separate group-holdout regimes. Direct next-event/task labels are usable only when observed by the primary split cutoff.
3,SPM0000004,SNP_SVC000004,SVC000004,MC000001,C000001,WS0003,2026-06-24 15:01:00,122774,TEST,2026-01-01 00:00:00,2026-08-03 17:33:00,2026-08-03 17:33:00,NaN,NaN,0,1,0,0,0,0,1,0,2026-08-03 17:33:00,TRAIN,0,0,TRAIN,0,0,6411767099563454294,5993427844424968809,SPLIT_MANIFEST_V1_2,42,SYNTHETIC,1.2.0,MAINTENANCE_EVENT_SIM_V1,PRIMARY=time-based; unseen motorcycle/workshop are separate group-holdout regimes. Direct next-event/task labels are usable only when observed by the primary split cutoff.
4,SPM0000005,SNP_SVC000005,SVC000005,MC000002,C000002,WS0008,2023-09-19 16:01:00,30196,TRAIN,2021-01-27 11:31:00,2025-06-30 23:59:59,2025-06-30 23:59:59,SVC000006,2023-11-13 14:24:00,1,0,1,0,1,1,1,1,2023-11-13 14:24:00,TRAIN,0,0,VALIDATION,1,0,4440362905931393030,15943022802158451407,SPLIT_MANIFEST_V1_2,42,SYNTHETIC,1.2.0,MAINTENANCE_EVENT_SIM_V1,PRIMARY=time-based; unseen motorcycle/workshop are separate group-holdout regimes. Direct next-event/task labels are usable only when observed by the primary split cutoff.


## 7. Dataset Envanter Tablosu

`table_meta` listesindeki her tablo için otomatik olarak shape, bellek
kullanımı, duplicate satır ve toplam eksik değer sayısı hesaplanıp tek bir
`dataset_inventory` DataFrame'inde toplanıyor.

In [28]:
inventory_rows = []
for meta in table_meta:
    df = dataframes[meta["key"]]
    inventory_rows.append({
        "dataset": meta["key"],
        "dataframe_name": meta["var"],
        "file_name": meta["file_name"],
        "file_type": meta["file_type"],
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1_000_000, 3),
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing_values": int(df.isna().sum().sum()),
    })

dataset_inventory = pd.DataFrame(inventory_rows)
display(dataset_inventory)

,dataset,dataframe_name,file_name,file_type,rows,columns,memory_mb,duplicate_rows,total_missing_values
0,workshops,df_workshops,workshops.csv,csv,10,35,0.016,0,0
1,customers,df_customers,customers.csv,csv,7000,16,5.730,0,13355
2,motorcycles,df_motorcycles,motorcycles.csv,csv,10000,29,14.521,0,19096
3,motorcycle_models,df_motorcycle_models,ridebase_motorcycle_models_v1.csv,csv,39,38,0.066,0,165
4,usage_profiles,df_usage_profiles,usage_profiles.csv,csv,10000,30,10.396,0,19096
5,mileage_monthly,df_mileage_monthly,mileage_timeline_monthly.csv,csv,344714,15,218.383,0,0
6,maintenance_tasks,df_maintenance_tasks,maintenance_tasks.csv,csv,98,17,0.092,0,306
7,maintenance_policies,df_maintenance_policies,maintenance_policies.csv,csv,621,30,0.662,0,6092
8,appointments,df_appointments,appointments.csv,csv,28150,13,24.438,0,3941
9,services,df_services,services.csv,csv,41518,43,74.269,0,431063


## 8. Sütun Veri Sözlüğü (`column_dictionary`)

Tüm tablolardaki tüm sütunlar tek bir DataFrame'de birleştiriliyor: her
dataset-column kombinasyonu için dtype, dolu/boş değer sayıları, boş değer
yüzdesi, benzersiz değer sayısı ve örnek bir değer gösteriliyor.

In [29]:
def get_example_value(series: pd.Series):
    non_null = series.dropna()
    if non_null.empty:
        return None
    value = str(non_null.iloc[0])
    return value if len(value) <= 200 else value[:200] + "..."


column_dictionary_rows = []
for meta in table_meta:
    df = dataframes[meta["key"]]
    n_rows = len(df)
    for col in df.columns:
        series = df[col]
        null_count = int(series.isna().sum())
        column_dictionary_rows.append({
            "dataset": meta["key"],
            "column_name": col,
            "dtype": str(series.dtype),
            "non_null_count": n_rows - null_count,
            "null_count": null_count,
            "null_pct": round((null_count / n_rows) * 100, 2) if n_rows else 0.0,
            "unique_count": int(series.nunique(dropna=True)),
            "example_value": get_example_value(series),
        })

column_dictionary = pd.DataFrame(column_dictionary_rows)
print("Toplam dataset-column kombinasyonu:", len(column_dictionary))
display(column_dictionary)

Toplam dataset-column kombinasyonu: 755


,dataset,column_name,dtype,non_null_count,null_count,null_pct,unique_count,example_value
0,workshops,workshop_id,object,10,0,0.0,10,WS0001
1,workshops,workshop_name,object,10,0,0.0,10,RideBase Sentetik Servis İstanbul Anadolu
2,workshops,city,object,10,0,0.0,10,İstanbul
3,workshops,district_profile,object,10,0,0.0,10,METROPOLITAN_DENSE
4,workshops,region,object,10,0,0.0,6,Marmara
...,...,...,...,...,...,...,...,...
750,split_manifest,random_seed,int64,41518,0,0.0,1,42
751,split_manifest,data_origin,object,41518,0,0.0,1,SYNTHETIC
752,split_manifest,generator_version,object,41518,0,0.0,1,1.2.0
753,split_manifest,scenario_id,object,41518,0,0.0,1,MAINTENANCE_EVENT_SIM_V1


In [30]:
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
column_dictionary.to_csv(REPORTS_TABLES_DIR / "column_dictionary.csv", index=False)
print("Kaydedildi:", REPORTS_TABLES_DIR / "column_dictionary.csv")

Kaydedildi: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/tables/column_dictionary.csv


## 9. ID / Key Sütunları (`key_columns_summary`)

Adında `_id` / `id` geçen sütunlar (kelime sınırına göre; "valid" gibi
sütunları yanlışlıkla yakalamamak için) otomatik olarak tespit ediliyor.
Ayrıca `task_code` da bir iş anahtarı (business key) olduğu için listeye
dahil ediliyor.

In [31]:
ID_PATTERN = re.compile(r"(^id$)|(^id_)|(_id$)|(_id_)", re.IGNORECASE)


def is_key_column(col: str) -> bool:
    return bool(ID_PATTERN.search(col)) or col == "task_code"


key_columns_rows = []
for meta in table_meta:
    df = dataframes[meta["key"]]
    for col in df.columns:
        if not is_key_column(col):
            continue
        series = df[col]
        unique_count = int(series.nunique(dropna=True))
        row_count = len(df)
        key_columns_rows.append({
            "dataset": meta["key"],
            "key_column": col,
            "unique_count": unique_count,
            "row_count": row_count,
            "is_unique": unique_count == row_count,
            "null_count": int(series.isna().sum()),
        })

key_columns_summary = pd.DataFrame(key_columns_rows)
display(key_columns_summary)

,dataset,key_column,unique_count,row_count,is_unique,null_count
0,workshops,workshop_id,10,10,True,0
1,workshops,scenario_id,10,10,True,0
2,customers,customer_id,7000,7000,True,0
3,customers,workshop_id,10,7000,False,0
4,motorcycles,motorcycle_id,10000,10000,True,0
5,motorcycles,customer_id,7000,10000,False,0
6,motorcycles,workshop_id,10,10000,False,0
7,motorcycles,model_id,39,10000,False,0
8,motorcycle_models,model_id,39,39,True,0
9,usage_profiles,usage_profile_id,10000,10000,True,0


Öne çıkan anahtar sütunlar (`service_id`, `motorcycle_id`, `customer_id`,
`workshop_id`, `appointment_id`, `service_task_id`, `part_id`, `snapshot_id`,
`task_code`) için hızlı bir filtre:

In [32]:
highlight_keys = [
    "service_id", "motorcycle_id", "customer_id", "workshop_id",
    "appointment_id", "service_task_id", "part_id", "snapshot_id", "task_code",
]
display(key_columns_summary[key_columns_summary["key_column"].isin(highlight_keys)])

,dataset,key_column,unique_count,row_count,is_unique,null_count
0,workshops,workshop_id,10,10,True,0
2,customers,customer_id,7000,7000,True,0
3,customers,workshop_id,10,7000,False,0
4,motorcycles,motorcycle_id,10000,10000,True,0
5,motorcycles,customer_id,7000,10000,False,0
6,motorcycles,workshop_id,10,10000,False,0
10,usage_profiles,motorcycle_id,10000,10000,True,0
11,usage_profiles,customer_id,7000,10000,False,0
12,usage_profiles,workshop_id,10,10000,False,0
14,mileage_monthly,motorcycle_id,10000,344714,False,0


## 10. Potansiyel Tarih Sütunları (`potential_datetime_columns`)

Adında `date`, `_at`, `month` veya `year` geçen sütunlar listeleniyor.
**Herhangi bir dönüşüm yapılmıyor** — bu sadece bir envanter, datetime'a
çevirme işlemi sonraki notebooklara bırakılıyor.

In [33]:
DATETIME_KEYWORDS = ["date", "_at", "month", "year"]


def looks_like_datetime_column(col: str) -> bool:
    lowered = col.lower()
    return any(keyword in lowered for keyword in DATETIME_KEYWORDS)


datetime_col_rows = []
for meta in table_meta:
    df = dataframes[meta["key"]]
    for col in df.columns:
        if looks_like_datetime_column(col):
            datetime_col_rows.append({
                "dataset": meta["key"],
                "column_name": col,
                "current_dtype": str(df[col].dtype),
            })

potential_datetime_columns = pd.DataFrame(datetime_col_rows)
print("Potansiyel tarih/zaman sütunu sayısı:", len(potential_datetime_columns))
display(potential_datetime_columns)

Potansiyel tarih/zaman sütunu sayısı: 63


,dataset,column_name,current_dtype
0,customers,first_seen_date,object
1,customers,churn_date,object
2,motorcycles,production_year,int64
3,motorcycles,production_year_basis,object
4,motorcycles,first_registration_date,object
5,motorcycles,ownership_start_date,object
6,motorcycles,observation_start_date,object
7,motorcycles,observation_end_date,object
8,motorcycles,warranty_months,int64
9,motorcycle_models,production_start_year,float64


## 11. JSON Metadata İncelemesi

### 11.1 `dataset_metadata.json`

In [34]:
print("Ana anahtarlar:")
print(list(dataset_metadata.keys()))

print("\n--- dataset bilgisi ---")
print(json.dumps(dataset_metadata["dataset"], indent=2, ensure_ascii=False))

print("\n--- coverage bilgisi (file_catalog ile düzeltilmiş authoritative satır sayıları) ---")
coverage_display = json.loads(json.dumps(dataset_metadata["coverage"]))
catalog_row_counts = {item["file_name"].replace(".csv", ""): item.get("row_count") for item in dataset_metadata.get("file_catalog", [])}
for entity_name in list(coverage_display.get("entity_counts", {})):
    if entity_name in catalog_row_counts:
        coverage_display["entity_counts"][entity_name] = catalog_row_counts[entity_name]
print(json.dumps(coverage_display, indent=2, ensure_ascii=False, default=str))

Ana anahtarlar:
['dataset', 'design_principles', 'prediction_tasks', 'coverage', 'ml_snapshot_contract', 'split_contract', 'provenance', 'file_catalog', 'expected_final_support_files', 'known_v1_notes', 'changelog_summary', 'null_semantics', 'layer_definitions', 'target_semantics', 'noise_audit_semantics', 'appointment_lifecycle', 'workshop_variability_decision', 'failure_generation_model']

--- dataset bilgisi ---
{
  "name": "RideBase Synthetic Motorcycle Maintenance Dataset",
  "short_name": "RideBase",
  "dataset_version": "1.2.0",
  "status": "V1_2_RELEASE_PASS",
  "generated_date": "2026-08-25",
  "language": "tr-TR",
  "currency": "TRY",
  "data_origin": "SYNTHETIC",
  "random_seed": 42,
  "generator_version": "1.2.0",
  "rule_set_version": "1.2.0",
  "scenario_id": "MAINTENANCE_EVENT_SIM_V1",
  "purpose": "Develop and evaluate leakage-aware machine-learning workflows for predicting the timing/kilometres of the next motorcycle service and the canonical maintenance tasks expected

In [35]:
entity_counts = dataset_metadata["coverage"]["entity_counts"].copy()
file_catalog_counts = {item["file_name"].replace(".csv", ""): item.get("row_count") for item in dataset_metadata.get("file_catalog", [])}
entity_counts.update({name: file_catalog_counts[name] for name in entity_counts if name in file_catalog_counts})
df_entity_counts = pd.DataFrame(
    list(entity_counts.items()), columns=["entity", "count"]
)
display(df_entity_counts)

,entity,count
0,workshops,10
1,customers,7000
2,motorcycles,10000
3,services,41518
4,service_tasks,204000
5,service_parts,76305
6,appointments,28150
7,canonical_maintenance_tasks,86
8,maintenance_policies,621
9,motorcycle_models,39


### 11.2 `quality_report.json`

In [36]:
release_gate = quality_report["report"]["release_gate"]
executive_summary = quality_report["executive_summary"]

print("release_gate:", release_gate)
print("total_checks:", executive_summary["total_checks"])
print("pass_checks:", executive_summary["pass_checks"])
print("warn_checks:", executive_summary.get("warn_checks", 0))
print("fail_checks:", executive_summary["fail_checks"])

release_gate: PASS
total_checks: 25
pass_checks: 25
warn_checks: 0
fail_checks: 0


In [37]:
df_quality_checks = pd.DataFrame(quality_report["quality_checks"])
display(df_quality_checks)

,check,status,severity,summary,metrics
0,full_row_duplicates,PASS,HIGH,No full-row duplicates are present.,"{'appointments': 0, 'customers': 0, 'maintenance_policies': 0, 'maintenance_tasks': 0, 'mileage_timeline_monthly': 0, 'motorcycles': 0, 'ridebase_motorcycle_models_v1': 0, 'service_parts': 0, 'ser..."
1,primary_key_duplicates,PASS,HIGH,All primary keys are unique.,"{'appointments': 0, 'customers': 0, 'maintenance_policies': 0, 'maintenance_tasks': 0, 'mileage_timeline_monthly': 0, 'motorcycles': 0, 'ridebase_motorcycle_models_v1': 0, 'service_parts': 0, 'ser..."
2,foreign_key_orphans,PASS,HIGH,All required foreign keys resolve.,"{'motorcycle_customer': 0, 'service_motorcycle': 0, 'service_customer': 0, 'service_workshop': 0, 'service_appointment': 0, 'task_service': 0, 'task_taxonomy': 0, 'part_service': 0, 'part_task': 0..."
3,canonical_timeline_monotonicity,PASS,HIGH,Canonical service timelines are monotonic.,"{'backwards_events': 0, 'violations': 0}"
4,service_task_status_consistency,PASS,HIGH,Task completion fields match task status.,"{'completed_bad': 0, 'declined_bad': 0, 'violations': 0}"
5,mileage_monotonicity,PASS,HIGH,Monthly odometer is monotonic.,"{'regressions': 0, 'violations': 0}"
6,service_part_compatibility,PASS,HIGH,Installed service parts remain compatible.,"{'incompatible_rows': 0, 'violations': 0}"
7,service_part_cost_arithmetic,PASS,HIGH,Part cost arithmetic reconciles.,"{'purchase_mismatch': 0, 'sale_mismatch': 0, 'violations': 0}"
8,appointment_lifecycle_validity,PASS,HIGH,Appointment lifecycle semantics remain valid.,"{'converted_without_service': 0, 'nonconverted_with_service': 0, 'violations': 0}"
9,parquet_readability,PASS,HIGH,All Parquets are readable with pyarrow.,"{'ml_maintenance_snapshots.parquet': {'rows': 41518, 'columns': 142}, 'ml_next_service_targets.parquet': {'rows': 41518, 'columns': 38}, 'ml_next_task_targets.parquet': {'rows': 41518, 'columns': ..."


# ML Dataset Yapısına İlk Bakış

`ml_maintenance_snapshots.parquet` model feature tablosudur.

`ml_next_service_targets.parquet` sonraki servis zamanı/kilometresi ve
right-censoring hedeflerini içerir.

`ml_next_task_targets.parquet` bir sonraki serviste gerçekleşecek kanonik
bakım görevlerinin multi-label hedeflerini içerir.

`split_manifest.csv` train/validation/test ve unseen motorcycle/workshop
ayrımlarını içerir.

Bu bölümde yalnızca shape, sütunlar ve temel key sütunlarının
tekilliği/kapsamı gösteriliyor. Feature leakage analizi bu notebookun
kapsamında **değildir**.

### ml_maintenance_snapshots.parquet

In [38]:
print("=== ml_maintenance_snapshots.parquet ===")
print("Shape:", df_ml_snapshots.shape)
print("\nSütunlar (tümü, {} adet):".format(df_ml_snapshots.shape[1]))
print(df_ml_snapshots.columns.tolist())

if "snapshot_id" in df_ml_snapshots.columns:
    print("\nsnapshot_id unique mi:", df_ml_snapshots["snapshot_id"].is_unique)
if "source_service_id" in df_ml_snapshots.columns:
    print("source_service_id unique mi:", df_ml_snapshots["source_service_id"].is_unique)
if "motorcycle_id" in df_ml_snapshots.columns:
    print("Unique motorcycle_id sayısı:", df_ml_snapshots["motorcycle_id"].nunique())
if "workshop_id" in df_ml_snapshots.columns:
    print("Unique workshop_id sayısı:", df_ml_snapshots["workshop_id"].nunique())

=== ml_maintenance_snapshots.parquet ===
Shape: (41518, 142)

Sütunlar (tümü, 142 adet):
['snapshot_id', 'source_service_id', 'snapshot_at', 'snapshot_date', 'service_sequence', 'snapshot_year', 'snapshot_month', 'snapshot_quarter', 'snapshot_day_of_year', 'snapshot_month_sin', 'snapshot_month_cos', 'motorcycle_id', 'customer_id', 'workshop_id', 'model_id', 'brand', 'model_name', 'category', 'powertrain_type', 'engine_displacement_cc', 'cylinder_count', 'cooling_type', 'final_drive_type', 'transmission_type', 'engine_oil_service_qty_l', 'spark_plug_count', 'fuel_type', 'policy_group', 'policy_ready', 'spec_confidence', 'production_year', 'motorcycle_age_years', 'ownership_months', 'initial_mileage_km', 'is_left_truncated', 'days_observed', 'customer_type', 'is_fleet_customer', 'acquisition_channel', 'usage_type', 'annual_km_baseline', 'city_ratio', 'highway_ratio', 'offroad_ratio', 'track_ratio', 'avg_ride_days_per_week', 'daily_active_probability', 'avg_km_per_active_day', 'riding_int

### ml_next_service_targets.parquet

In [39]:
print("=== ml_next_service_targets.parquet ===")
print("Shape:", df_next_service_targets.shape)
print("\nSütunlar (tümü, {} adet):".format(df_next_service_targets.shape[1]))
print(df_next_service_targets.columns.tolist())

if "snapshot_id" in df_next_service_targets.columns:
    print("\nsnapshot_id unique mi:", df_next_service_targets["snapshot_id"].is_unique)
if "source_service_id" in df_next_service_targets.columns:
    print("source_service_id unique mi:", df_next_service_targets["source_service_id"].is_unique)
if "motorcycle_id" in df_next_service_targets.columns:
    print("Unique motorcycle_id sayısı:", df_next_service_targets["motorcycle_id"].nunique())
if "workshop_id" in df_next_service_targets.columns:
    print("Unique workshop_id sayısı:", df_next_service_targets["workshop_id"].nunique())

=== ml_next_service_targets.parquet ===
Shape: (41518, 38)

Sütunlar (tümü, 38 adet):
['snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'snapshot_at', 'snapshot_odometer_km', 'target_event_observed', 'is_right_censored', 'next_service_id', 'next_service_received_at_raw', 'target_next_service_at', 'next_service_delivered_at', 'days_to_next_service_raw', 'days_to_next_service', 'target_time_repaired', 'next_service_odometer_km', 'km_to_next_service_raw', 'km_to_next_service', 'target_km_valid', 'next_service_mileage_source', 'next_service_mileage_quality_flag', 'next_service_is_mileage_estimated', 'next_service_type_code', 'next_service_primary_trigger_task', 'next_service_arrival_mode', 'next_service_is_breakdown', 'next_service_is_warranty', 'censor_at', 'censor_days', 'days_to_event_or_censor', 'censor_source', 'censor_time_repaired', 'target_version', 'data_origin', 'generator_version', 'random_seed', 'scenario_id']

snapshot_id unique mi: True
sourc

### ml_next_task_targets.parquet

In [40]:
print("=== ml_next_task_targets.parquet ===")
print("Shape:", df_next_task_targets.shape)
print("\nSütunlar (tümü, {} adet):".format(df_next_task_targets.shape[1]))
print(df_next_task_targets.columns.tolist())

if "snapshot_id" in df_next_task_targets.columns:
    print("\nsnapshot_id unique mi:", df_next_task_targets["snapshot_id"].is_unique)
if "source_service_id" in df_next_task_targets.columns:
    print("source_service_id unique mi:", df_next_task_targets["source_service_id"].is_unique)
if "motorcycle_id" in df_next_task_targets.columns:
    print("Unique motorcycle_id sayısı:", df_next_task_targets["motorcycle_id"].nunique())
if "workshop_id" in df_next_task_targets.columns:
    print("Unique workshop_id sayısı:", df_next_task_targets["workshop_id"].nunique())

=== ml_next_task_targets.parquet ===
Shape: (41518, 122)

Sütunlar (tümü, 122 adet):
['snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'next_service_id', 'target_event_observed', 'is_right_censored', 'task__ENGINE_OIL_CHECK', 'task__ENGINE_OIL_CHANGE', 'task__OIL_FILTER_CHANGE', 'task__OIL_LEAK_INSPECTION', 'task__ENGINE_COMPRESSION_TEST', 'task__VALVE_CLEARANCE_INSPECTION', 'task__VALVE_CLEARANCE_ADJUST', 'task__CAM_CHAIN_INSPECTION', 'task__CAM_CHAIN_TENSIONER_SERVICE', 'task__AIR_FILTER_INSPECTION', 'task__AIR_FILTER_CLEAN', 'task__AIR_FILTER_CHANGE', 'task__THROTTLE_BODY_CLEAN', 'task__INTAKE_LEAK_INSPECTION', 'task__SPARK_PLUG_INSPECTION', 'task__SPARK_PLUG_CHANGE', 'task__FUEL_FILTER_INSPECTION', 'task__FUEL_FILTER_CHANGE', 'task__FUEL_INJECTOR_CLEAN', 'task__FUEL_SYSTEM_DIAGNOSTIC', 'task__COOLANT_LEVEL_CHECK', 'task__COOLANT_CHANGE', 'task__COOLING_SYSTEM_INSPECTION', 'task__RADIATOR_CLEAN', 'task__THERMOSTAT_DIAGNOSTIC', 'task__CLUTCH_FREE_PLA

### split_manifest.csv

In [41]:
print("=== split_manifest.csv ===")
print("Shape:", df_split_manifest.shape)
print("\nSütunlar (tümü, {} adet):".format(df_split_manifest.shape[1]))
print(df_split_manifest.columns.tolist())

if "snapshot_id" in df_split_manifest.columns:
    print("\nsnapshot_id unique mi:", df_split_manifest["snapshot_id"].is_unique)
if "source_service_id" in df_split_manifest.columns:
    print("source_service_id unique mi:", df_split_manifest["source_service_id"].is_unique)
if "motorcycle_id" in df_split_manifest.columns:
    print("Unique motorcycle_id sayısı:", df_split_manifest["motorcycle_id"].nunique())
if "workshop_id" in df_split_manifest.columns:
    print("Unique workshop_id sayısı:", df_split_manifest["workshop_id"].nunique())

=== split_manifest.csv ===
Shape: (41518, 37)

Sütunlar (tümü, 37 adet):
['manifest_id', 'snapshot_id', 'source_service_id', 'motorcycle_id', 'customer_id', 'workshop_id', 'snapshot_at', 'snapshot_odometer_km', 'primary_time_split', 'primary_split_start_at', 'primary_split_end_at', 'primary_label_cutoff_at', 'next_service_id', 'next_service_at', 'full_dataset_event_observed', 'full_dataset_right_censored', 'event_observed_by_primary_cutoff', 'boundary_crossing_future_target', 'task_target_eligible_primary', 'next_service_regression_eligible_primary', 'survival_target_eligible_primary', 'survival_event_observed_in_window', 'survival_admin_censor_at', 'unseen_motorcycle_split', 'is_unseen_motorcycle_validation', 'is_unseen_motorcycle_test', 'unseen_workshop_split', 'is_unseen_workshop_validation', 'is_unseen_workshop_test', 'motorcycle_group_hash', 'workshop_group_hash', 'split_version', 'random_seed', 'data_origin', 'generator_version', 'scenario_id', 'notes']

snapshot_id unique mi: Tr

# İlk İnceleme Özeti

In [42]:
total_dataframes = len(dataframes)
total_rows = sum(df.shape[0] for df in dataframes.values())
total_dataset_columns = len(column_dictionary)

n_motorcycles = df_motorcycles["motorcycle_id"].nunique()
n_customers = df_customers["customer_id"].nunique()
n_services = df_services["service_id"].nunique()
n_service_tasks = len(df_service_tasks)
n_service_parts = len(df_service_parts)
n_snapshots = df_ml_snapshots["snapshot_id"].nunique()

print("Toplam dataframe sayısı:", total_dataframes)
print("Toplam satır sayısı:", total_rows)
print("Toplam dataset-column sayısı:", total_dataset_columns)
print("Motosiklet sayısı:", n_motorcycles)
print("Müşteri sayısı:", n_customers)
print("Servis sayısı:", n_services)
print("Service task sayısı:", n_service_tasks)
print("Service part sayısı:", n_service_parts)
print("Snapshot sayısı:", n_snapshots)

Toplam dataframe sayısı: 19
Toplam satır sayısı: 1289310
Toplam dataset-column sayısı: 755
Motosiklet sayısı: 10000
Müşteri sayısı: 7000
Servis sayısı: 41518
Service task sayısı: 204000
Service part sayısı: 76305
Snapshot sayısı: 41518


Bir sonraki aşama: `02_eda.ipynb`

# 02 — Veri Kalitesi ve Temizleme

Bu bölümde eksik değerler, tekrarlar, boş metinler, sonsuz değerler ve sabit sütunlar incelenir.
Henüz veri üzerinde değişiklik yapılmaz.

In [43]:
# ============================================================
# 02.1 — Genel veri kalitesi kontrolü
# ============================================================

def create_quality_report(dataframes: dict[str, pd.DataFrame]):
    """
    Tüm dataframe'ler için genel kalite özeti üretir.
    Veriler üzerinde değişiklik yapmaz.
    """
    dataset_rows = []
    missing_rows = []
    suspicious_rows = []

    for dataset_name, df in dataframes.items():
        row_count, column_count = df.shape

        # Tamamen aynı olan satırlar
        duplicate_count = int(df.duplicated().sum())

        # Eksik değerler
        total_missing = int(df.isna().sum().sum())
        total_cells = row_count * column_count
        missing_pct = (
            100 * total_missing / total_cells
            if total_cells > 0 else 0
        )

        # Sayısal sütunlardaki +/- infinity değerleri
        numeric_df = df.select_dtypes(include=np.number)
        infinity_count = (
            int(np.isinf(numeric_df.to_numpy(dtype="float64", na_value=np.nan)).sum())
            if not numeric_df.empty else 0
        )

        # Metin sütunlarındaki boş veya sadece boşluk içeren değerler
        text_columns = df.select_dtypes(include=["object", "string"]).columns
        blank_string_count = 0

        for column in text_columns:
            blank_count = int(
                df[column]
                .astype("string")
                .str.strip()
                .eq("")
                .sum()
            )

            blank_string_count += blank_count

            if blank_count > 0:
                suspicious_rows.append({
                    "dataset": dataset_name,
                    "column": column,
                    "issue": "blank_string",
                    "count": blank_count,
                    "pct": round(100 * blank_count / row_count, 2)
                    if row_count else 0,
                })

        # Sütun bazında eksiklik ve sabit sütun kontrolü
        for column in df.columns:
            null_count = int(df[column].isna().sum())
            unique_count = int(df[column].nunique(dropna=True))

            if null_count > 0:
                missing_rows.append({
                    "dataset": dataset_name,
                    "column": column,
                    "dtype": str(df[column].dtype),
                    "null_count": null_count,
                    "null_pct": round(
                        100 * null_count / row_count, 2
                    ) if row_count else 0,
                    "unique_non_null": unique_count,
                })

            if unique_count == 0:
                suspicious_rows.append({
                    "dataset": dataset_name,
                    "column": column,
                    "issue": "all_missing",
                    "count": row_count,
                    "pct": 100.0,
                })

            elif unique_count == 1:
                suspicious_rows.append({
                    "dataset": dataset_name,
                    "column": column,
                    "issue": "constant_column",
                    "count": row_count - null_count,
                    "pct": round(
                        100 * (row_count - null_count) / row_count, 2
                    ) if row_count else 0,
                })

        dataset_rows.append({
            "dataset": dataset_name,
            "rows": row_count,
            "columns": column_count,
            "duplicate_rows": duplicate_count,
            "duplicate_pct": round(
                100 * duplicate_count / row_count, 4
            ) if row_count else 0,
            "total_missing": total_missing,
            "missing_pct": round(missing_pct, 4),
            "blank_strings": blank_string_count,
            "infinity_values": infinity_count,
        })

    dataset_quality = (
        pd.DataFrame(dataset_rows)
        .sort_values(
            ["duplicate_rows", "missing_pct"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    missing_quality = (
        pd.DataFrame(missing_rows)
        .sort_values(
            ["null_pct", "null_count"],
            ascending=False
        )
        .reset_index(drop=True)
        if missing_rows
        else pd.DataFrame(
            columns=[
                "dataset", "column", "dtype",
                "null_count", "null_pct", "unique_non_null"
            ]
        )
    )

    suspicious_quality = (
        pd.DataFrame(suspicious_rows)
        .sort_values(
            ["issue", "pct"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
        if suspicious_rows
        else pd.DataFrame(
            columns=["dataset", "column", "issue", "count", "pct"]
        )
    )

    return dataset_quality, missing_quality, suspicious_quality


dataset_quality, missing_quality, suspicious_quality = (
    create_quality_report(dataframes)
)

def conditional_missing_violations(dataframes: dict[str, pd.DataFrame]) -> pd.DataFrame:
    violations = []

    def add(dataset, column, rule, mask):
        count = int(pd.Series(mask).fillna(False).sum())
        violations.append({
            "dataset": dataset,
            "column": column,
            "rule": rule,
            "violation_count": count,
            "status": "PASS" if count == 0 else "FAIL",
        })

    customers = dataframes["customers"]
    add("customers", "churn_date", "is_active=0 ise zorunlu",
        customers["is_active"].eq(0) & customers["churn_date"].isna())

    motorcycles = dataframes["motorcycles"]
    add("motorcycles", "observation_end_date", "is_active=0 ise zorunlu",
        motorcycles["is_active"].eq(0) & motorcycles["observation_end_date"].isna())

    usage = dataframes["usage_profiles"][["motorcycle_id", "profile_end_date"]].merge(
        motorcycles[["motorcycle_id", "is_active", "observation_end_date"]],
        on="motorcycle_id", how="left"
    )
    add("usage_profiles", "profile_end_date", "pasif motosiklette observation_end_date ile eşit olmalı",
        usage["is_active"].eq(0) & (
            pd.to_datetime(usage["profile_end_date"], errors="coerce") !=
            pd.to_datetime(usage["observation_end_date"], errors="coerce")
        ))

    models = dataframes["motorcycle_models"]
    add("motorcycle_models", "production_start_year", "tüm modellerde zorunlu",
        models["production_start_year"].isna())
    add("motorcycle_models", "production_end_year", "tüm modellerde zorunlu",
        models["production_end_year"].isna())

    noise = dataframes["noise_audit"]
    add("noise_audit", "canonical_clean_value", "clean_value_available=1 ise zorunlu",
        noise["clean_value_available"].eq(1) & noise["canonical_clean_value"].isna())
    add("noise_audit", "original_clean_value", "original_clean_value_available=1 ise zorunlu",
        noise["original_clean_value_available"].eq(1) & noise["original_clean_value"].isna())

    history = dataframes["service_status_history"]
    add("service_status_history", "repair_reason", "was_timestamp_repaired=1 ise zorunlu",
        history["was_timestamp_repaired"].eq(1) & history["repair_reason"].isna())

    appointments = dataframes["appointments"]
    add("appointments", "service_id", "CONVERTED_TO_SERVICE ise zorunlu",
        appointments["status"].eq("CONVERTED_TO_SERVICE") & appointments["service_id"].isna())

    next_service = dataframes["next_service_targets"]
    observed_service = next_service["target_event_observed"].eq(1)
    for column in ["next_service_id", "target_next_service_at", "days_to_next_service"]:
        add("next_service_targets", column, "event gözlenmişse zorunlu",
            observed_service & next_service[column].isna())

    next_task = dataframes["next_task_targets"]
    task_columns = [c for c in next_task.columns if c.startswith("task__")]
    observed_task = next_task["target_event_observed"].eq(1)
    censored_task = next_task["target_event_observed"].eq(0)
    add("next_task_targets", "task__*", "gözlenen eventte label NULL olamaz",
        observed_task & next_task[task_columns].isna().any(axis=1))
    add("next_task_targets", "task__*", "censored eventte label NULL olmalı",
        censored_task & next_task[task_columns].notna().any(axis=1))

    return pd.DataFrame(violations)


def explain_missing_semantics(row):
    dataset, column = row["dataset"], row["column"]
    if column == "notes" or column.endswith("_notes"):
        return "Opsiyonel serbest metin"
    if dataset == "services" and column in {"labor_total", "parts_total", "discount_total", "tax_total", "grand_total"}:
        return "Base event katmanında bilinçli boş; services_enriched kullanılmalı"
    if dataset in {"customers", "motorcycles", "usage_profiles"} and column in {"churn_date", "observation_end_date", "profile_end_date"}:
        return "Aktif kayıtlarda beklenen koşullu NULL"
    if dataset == "maintenance_policies":
        return "Policy kind/scope/trigger türüne bağlı koşullu NULL"
    if dataset == "maintenance_tasks" and column.startswith("required_"):
        return "Task uygulanabilirliğine bağlı koşullu NULL"
    if dataset == "motorcycle_models":
        return "Teknik özellik kaynakta bilinmiyor; sahte değer üretilmedi"
    if dataset == "noise_audit":
        return "Availability flag ile tanımlanan koşullu audit NULL'ı"
    if dataset == "service_status_history":
        return "State/repair durumuna bağlı koşullu NULL"
    if dataset in {"next_service_targets", "next_task_targets"}:
        return "Right-censoring veya boş task listesi nedeniyle beklenen NULL"
    if dataset == "ml_snapshots":
        return "Bilinmeyen teknik/geçmiş feature; sentinel yerine gerçek NULL"
    if dataset == "appointments" and column == "service_id":
        return "CANCELLED/NO_SHOW/RESCHEDULED randevularda beklenen NULL"
    if dataset in {"services", "services_enriched", "service_tasks", "service_parts"}:
        return "Operasyon/task durumuna bağlı opsiyonel alan"
    return "İncelenmeli"


missing_violations = conditional_missing_violations(dataframes)
missing_semantics = missing_quality.copy()
missing_semantics["semantic_reason"] = missing_semantics.apply(explain_missing_semantics, axis=1)
missing_semantics["semantic_status"] = np.where(
    missing_semantics["semantic_reason"].eq("İncelenmeli"), "REVIEW", "EXPECTED/CONDITIONAL"
)

print("=== DATASET BAZINDA GENEL KALİTE ===")
display(dataset_quality)

print("\n=== KURAL İHLALİ OLAN EKSİKLER ===")
display(missing_violations)
print("Toplam eksik-değer kural ihlali:", int(missing_violations["violation_count"].sum()))

print("\n=== %50 ÜZERİ BEKLENEN / KOŞULLU NULL'LAR ===")
display(missing_semantics.loc[
    missing_semantics["null_pct"].gt(50),
    ["dataset", "column", "null_pct", "semantic_status", "semantic_reason"]
].sort_values("null_pct", ascending=False))

print("\n=== DİĞER ŞÜPHELİ SÜTUNLAR ===")
display(suspicious_quality)

=== DATASET BAZINDA GENEL KALİTE ===


,dataset,rows,columns,duplicate_rows,duplicate_pct,total_missing,missing_pct,blank_strings,infinity_values
0,maintenance_policies,621,30,0,0.0,6092,32.6999,0,0
1,services,41518,43,0,0.0,431063,24.1455,0,0
2,maintenance_tasks,98,17,0,0.0,306,18.3673,0,0
3,next_task_targets,41518,122,0,0.0,918351,18.1306,0,0
4,customers,7000,16,0,0.0,13355,11.9241,0,0
5,motorcycle_models,39,38,0,0.0,165,11.1336,0,0
6,services_enriched,41518,49,0,0.0,223473,10.9848,0,0
7,next_service_targets,41518,38,0,0.0,160398,10.1667,0,0
8,motorcycles,10000,29,0,0.0,19096,6.5848,0,0
9,usage_profiles,10000,30,0,0.0,19096,6.3653,0,0



=== KURAL İHLALİ OLAN EKSİKLER ===


,dataset,column,rule,violation_count,status
0,customers,churn_date,is_active=0 ise zorunlu,0,PASS
1,motorcycles,observation_end_date,is_active=0 ise zorunlu,0,PASS
2,usage_profiles,profile_end_date,pasif motosiklette observation_end_date ile eşit olmalı,0,PASS
3,motorcycle_models,production_start_year,tüm modellerde zorunlu,0,PASS
4,motorcycle_models,production_end_year,tüm modellerde zorunlu,0,PASS
5,noise_audit,canonical_clean_value,clean_value_available=1 ise zorunlu,0,PASS
6,noise_audit,original_clean_value,original_clean_value_available=1 ise zorunlu,0,PASS
7,service_status_history,repair_reason,was_timestamp_repaired=1 ise zorunlu,0,PASS
8,appointments,service_id,CONVERTED_TO_SERVICE ise zorunlu,0,PASS
9,next_service_targets,next_service_id,event gözlenmişse zorunlu,0,PASS


Toplam eksik-değer kural ihlali: 0

=== %50 ÜZERİ BEKLENEN / KOŞULLU NULL'LAR ===


,dataset,column,null_pct,semantic_status,semantic_reason
0,noise_audit,original_clean_value,100.00,EXPECTED/CONDITIONAL,Availability flag ile tanımlanan koşullu audit NULL'ı
6,services,grand_total,100.00,EXPECTED/CONDITIONAL,Base event katmanında bilinçli boş; services_enriched kullanılmalı
1,services,technician_notes,100.00,EXPECTED/CONDITIONAL,Opsiyonel serbest metin
10,customers,notes,100.00,EXPECTED/CONDITIONAL,Opsiyonel serbest metin
8,motorcycles,notes,100.00,EXPECTED/CONDITIONAL,Opsiyonel serbest metin
7,services_enriched,technician_notes,100.00,EXPECTED/CONDITIONAL,Opsiyonel serbest metin
9,usage_profiles,notes,100.00,EXPECTED/CONDITIONAL,Opsiyonel serbest metin
5,services,tax_total,100.00,EXPECTED/CONDITIONAL,Base event katmanında bilinçli boş; services_enriched kullanılmalı
4,services,discount_total,100.00,EXPECTED/CONDITIONAL,Base event katmanında bilinçli boş; services_enriched kullanılmalı
3,services,parts_total,100.00,EXPECTED/CONDITIONAL,Base event katmanında bilinçli boş; services_enriched kullanılmalı



=== DİĞER ŞÜPHELİ SÜTUNLAR ===


,dataset,column,issue,count,pct
0,customers,notes,all_missing,7000,100.00
1,motorcycles,notes,all_missing,10000,100.00
2,usage_profiles,notes,all_missing,10000,100.00
3,services,technician_notes,all_missing,41518,100.00
4,services,labor_total,all_missing,41518,100.00
5,services,parts_total,all_missing,41518,100.00
6,services,discount_total,all_missing,41518,100.00
7,services,tax_total,all_missing,41518,100.00
8,services,grand_total,all_missing,41518,100.00
9,services_enriched,technician_notes,all_missing,41518,100.00
